# Comprehensive PCA Analysis: Unleashing the Full Potential

This notebook demonstrates the complete power of Principal Component Analysis (PCA) through a city temperature dataset. We'll cover:

1. **Fundamentals** - Variance decomposition, eigenstructure, geometric interpretation
2. **Insights & Interpretation** - Loading analysis, biplots, relating PCs to domain knowledge
3. **Discovering Causes of Variation** - Connecting components to real-world factors
4. **Anomaly Detection** - Hotelling's T², Q-residuals (SPE), multivariate control charts
5. **Data Imputation** - Using PCA to fill missing values
6. **Clustering in PC Space** - Leveraging dimensionality reduction for clustering
7. **Orthogonal Decomposition** - Understanding independent sources of variation
8. **Leave-One-Out Validation** - Assessing model stability and influence
9. **Advanced Diagnostics** - Contribution plots, reconstruction errors, leverage analysis

### Cell 1: Library Imports and Environment Setup

This cell imports all the necessary Python libraries for our comprehensive PCA analysis:

**Core Scientific Computing:**
- `numpy`: Fundamental package for numerical operations, matrix computations, and linear algebra
- `pandas`: Data manipulation and analysis library for handling tabular data (DataFrames)
- `scipy.stats`: Statistical functions including correlation tests, probability distributions, and linear regression

**Visualization Libraries:**
- `matplotlib.pyplot`: Base plotting library for static visualizations
- `seaborn`: Statistical visualization library built on matplotlib with aesthetically pleasing defaults
- `plotly.express` and `plotly.graph_objects`: Interactive visualization libraries that allow zooming, panning, and hovering over data points

**Machine Learning (scikit-learn):**
- `PCA`: The core Principal Component Analysis implementation
- `StandardScaler`: Standardizes features by removing the mean and scaling to unit variance (z-score normalization)
- `KMeans`, `AgglomerativeClustering`: Clustering algorithms for grouping similar observations
- `silhouette_score`, `calinski_harabasz_score`: Metrics for evaluating cluster quality
- `SimpleImputer`: For handling missing values

**Important Notes:**
- We suppress warnings to keep the output clean
- We set a consistent visual style (`seaborn-v0_8-whitegrid`) for all matplotlib plots
- The `husl` color palette provides perceptually uniform colors that are distinguishable

In [3]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.impute import SimpleImputer

# Interactive plotting
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Libraries loaded successfully!")

Libraries loaded successfully!


---
## Part 1: Data Loading and Exploration

### Cell 2: Loading the Dataset

This cell loads our dataset from an Excel file using pandas' `read_excel()` function (which requires the `openpyxl` package for `.xlsx` files).

**Dataset Overview:**
The CityTemp dataset contains monthly average temperatures for various European cities. The data structure is:
- **Rows**: Each row represents a different city
- **Columns**: 
  - `Cities`: Name of the city (categorical identifier)
  - `Jan` through `Dec`: Monthly average temperatures in degrees Celsius (12 numerical features)
  - `Latitude`: Geographic latitude of the city in degrees North

**Why This Dataset is Perfect for PCA:**
1. **Multiple correlated variables**: Monthly temperatures are highly correlated (adjacent months tend to have similar values)
2. **Clear underlying structure**: We expect temperature patterns to be explained by latitude and continentality
3. **Interpretable results**: We can validate PCA findings against known geographic/climatic facts

The `df.head(10)` command displays the first 10 rows to give us an initial look at the data structure and values.

In [4]:
# Load the data
df = pd.read_excel('CityTemp.xlsx')
print(f"Dataset shape: {df.shape}")
print(f"Cities: {df.shape[0]}")
print(f"Features: {df.shape[1] - 2} months + latitude")
df.head(10)

Dataset shape: (33, 14)
Cities: 33
Features: 12 months + latitude


,Cities,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Latitude
0,Amsterdam,2.0,2.0,5.0,8.0,12.0,15.0,17.0,17.0,15.0,11.0,6.0,3.0,52.37
1,Ankara,1.0,0.0,6.0,12.0,17.0,19.0,23.0,22.0,18.0,14.0,8.0,2.0,39.87
2,Athens,9.0,10.0,12.0,16.0,21.0,26.0,29.0,28.0,24.0,20.0,15.0,11.0,37.97
3,Belgrade,1.0,3.0,8.0,13.0,17.0,22.0,23.0,23.0,19.0,14.0,8.0,6.0,44.82
4,Berlin,-2.0,1.0,4.0,10.0,14.0,18.0,21.0,19.0,16.0,10.0,4.0,0.0,52.52
5,Bern,1.0,1.0,6.0,10.0,13.0,17.0,18.0,19.0,15.0,10.0,4.0,1.0,46.95
6,Bonn,1.0,2.0,7.0,11.0,14.0,18.0,20.0,16.0,16.0,11.0,7.0,2.0,50.73
7,Bruxells,1.0,4.0,7.0,11.0,13.0,18.0,19.0,18.0,17.0,12.0,7.0,3.0,50.85
8,Bucarest,-4.0,-2.0,6.0,12.0,17.0,21.0,23.0,23.0,18.0,13.0,7.0,1.0,44.42
9,Budapest,-2.0,1.0,7.0,13.0,17.0,21.0,23.0,23.0,18.0,13.0,6.0,1.0,47.47


### Cell 3: Data Preparation and Variable Definition

This cell organizes our data by defining clear variable groups and extracting the numerical matrix for analysis.

**Column Definitions:**
- `city_col`: The column name containing city identifiers (used for labeling, not analysis)
- `month_cols`: A list of the 12 month abbreviations—these are our **features** or **variables** that PCA will analyze
- `cities`: A Python list of city names for convenient access during plotting and interpretation

**Data Extraction:**
- `X_raw`: The **data matrix** of shape $(n \times p)$ where:
  - $n$ = number of observations (cities) = 33
  - $p$ = number of features (months) = 12
  - Each element $X_{ij}$ represents the average temperature in city $i$ during month $j$
- `latitudes`: An auxiliary variable (not used in PCA) that will help us interpret results

**Terminology Note:**
In PCA literature, you'll encounter different naming conventions:
- **Observations** = samples = rows = cities
- **Variables** = features = columns = months
- **Data matrix X**: rows are observations, columns are variables

In [5]:
# Define column groups
city_col = 'Cities'
month_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
cities = df[city_col].tolist()

# Extract temperature matrix (X) and metadata
X_raw = df[month_cols].values
latitudes = df['Latitude'].values

print(f"Temperature matrix shape: {X_raw.shape}")
print(f"Temperature range: {X_raw.min():.1f}°C to {X_raw.max():.1f}°C")

Temperature matrix shape: (33, 12)
Temperature range: -14.0°C to 30.0°C


### Cell 4: Visualizing Raw Temperature Profiles

This cell creates an **interactive line plot** showing the temperature profile (temperature vs. month) for each city using Plotly.

**Why Visualize Raw Data First?**
Before applying any statistical technique, it's crucial to visually inspect the data to:
1. **Identify patterns**: Look for common trends across cities
2. **Spot outliers**: Cities with unusual temperature patterns
3. **Understand variability**: How much do cities differ from each other?
4. **Form hypotheses**: What might the principal components represent?

**What to Look For:**
- **Seasonal pattern**: All cities should show higher temperatures in summer (Jun-Aug) and lower in winter (Dec-Feb) for Northern Hemisphere
- **Vertical spread**: The vertical distance between lines at any month shows inter-city variability
- **Amplitude differences**: Some cities have a larger summer-winter difference (continental) than others (maritime)
- **Parallel patterns**: Lines that are roughly parallel suggest a common underlying structure

**Interactive Features:**
- Hover over points to see exact temperature values
- Click on legend items to toggle city visibility
- Use the zoom and pan tools to focus on specific regions

In [6]:
# Visualize the raw data: Temperature profiles
fig = go.Figure()

for i, city in enumerate(cities):
    fig.add_trace(go.Scatter(
        x=month_cols, y=X_raw[i], mode='lines+markers',
        name=city, hovertemplate=f"<b>{city}</b><br>%{{x}}: %{{y}}°C<extra></extra>"
    ))

fig.update_layout(
    title='Monthly Temperature Profiles by City',
    xaxis_title='Month', yaxis_title='Temperature (°C)',
    height=600, width=1000,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5)
)
fig.show()

### Cell 5: Correlation Matrix Heatmap

This cell computes and visualizes the **Pearson correlation matrix** between all pairs of months.

**What is the Correlation Matrix?**
The correlation matrix $\mathbf{R}$ is a symmetric $p \times p$ matrix where:
- $R_{ij}$ = Pearson correlation coefficient between month $i$ and month $j$
- Diagonal elements are always 1 (each variable correlates perfectly with itself)
- Values range from -1 (perfect negative correlation) to +1 (perfect positive correlation)

**Mathematical Definition:**
$$R_{ij} = \frac{\text{Cov}(X_i, X_j)}{\sigma_{X_i} \cdot \sigma_{X_j}} = \frac{\sum_{k=1}^{n}(X_{ki} - \bar{X}_i)(X_{kj} - \bar{X}_j)}{\sqrt{\sum_{k=1}^{n}(X_{ki} - \bar{X}_i)^2} \cdot \sqrt{\sum_{k=1}^{n}(X_{kj} - \bar{X}_j)^2}}$$

**Why is This Important for PCA?**
- PCA finds linear combinations of variables that capture maximum variance
- **High correlations** indicate redundancy—PCA will compress this
- The correlation matrix is actually the **covariance matrix of standardized data**
- PCA performed on standardized data essentially does eigendecomposition of this correlation matrix

**Expected Patterns:**
- Adjacent months should be highly correlated (e.g., Jan-Feb, Jun-Jul)
- Summer and winter months should have negative or weak correlations
- This block structure suggests a strong first principal component

In [7]:
# Correlation heatmap of months
corr_matrix = pd.DataFrame(X_raw, columns=month_cols).corr()

fig = px.imshow(corr_matrix, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Month-to-Month Temperature Correlation')
fig.update_layout(width=700, height=600)
fig.show()

**Insight:** Notice strong correlations between adjacent months and anti-correlation between summer and winter months. This suggests a strong seasonal pattern that PCA should capture in the first component.

In [ ]:
# The following code provides a dropdown menu to choose three months and visualize 
# the 3D scatter plot of cities based on their temperatures in those months.

import ipywidgets as widgets
from IPython.display import display

def plot_3d_temperatures(month_x, month_y, month_z):
    """Create an interactive 3D scatter plot of cities based on selected months."""
    fig = go.Figure(data=[go.Scatter3d(
        x=df[month_x],
        y=df[month_y],
        z=df[month_z],
        mode='markers+text',
        text=cities,
        textposition='top center',
        marker=dict(
            size=8,
            color=latitudes,
            colorscale='Viridis',
            colorbar=dict(title='Latitude (°N)'),
            opacity=0.8
        ),
        hovertemplate=(
            '<b>%{text}</b><br>' +
            f'{month_x}: %{{x:.1f}}°C<br>' +
            f'{month_y}: %{{y:.1f}}°C<br>' +
            f'{month_z}: %{{z:.1f}}°C<br>' +
            'Latitude: %{marker.color:.1f}°N<extra></extra>'
        )
    )])
    
    fig.update_layout(
        title=f'3D Temperature Visualization: {month_x} vs {month_y} vs {month_z}',
        scene=dict(
            xaxis_title=f'{month_x} Temperature (°C)',
            yaxis_title=f'{month_y} Temperature (°C)',
            zaxis_title=f'{month_z} Temperature (°C)',
        ),
        width=900,
        height=700
    )
    fig.show()

# Create dropdown widgets for month selection
month_x_dropdown = widgets.Dropdown(
    options=month_cols,
    value='Jan',
    description='X-axis:',
    style={'description_width': 'initial'}
)

month_y_dropdown = widgets.Dropdown(
    options=month_cols,
    value='Jul',
    description='Y-axis:',
    style={'description_width': 'initial'}
)

month_z_dropdown = widgets.Dropdown(
    options=month_cols,
    value='Apr',
    description='Z-axis:',
    style={'description_width': 'initial'}
)

# Create interactive output
interactive_plot = widgets.interactive_output(
    plot_3d_temperatures,
    {'month_x': month_x_dropdown, 'month_y': month_y_dropdown, 'month_z': month_z_dropdown}
)

# Display the widgets and plot
print("Select three months to visualize cities in 3D temperature space:")
display(widgets.HBox([month_x_dropdown, month_y_dropdown, month_z_dropdown]))
display(interactive_plot)

Select three months to visualize cities in 3D temperature space:


Output()

---
## Part 2: PCA Fundamentals - The Mathematics

### Cell 6: Data Standardization (Z-Score Normalization)

This cell standardizes the data using **z-score normalization**, which is a **critical preprocessing step** for PCA.

**What is Standardization?**
For each variable (column) $j$, we transform the values:
$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$

Where:
- $x_{ij}$ = original value for observation $i$, variable $j$
- $\bar{x}_j$ = mean of variable $j$
- $s_j$ = standard deviation of variable $j$
- $z_{ij}$ = standardized value (z-score)

**After Standardization:**
- Each variable has **mean = 0** (centered)
- Each variable has **standard deviation = 1** (scaled)

**Why is Standardization Crucial for PCA?**
1. **Scale invariance**: Without standardization, variables with larger absolute values would dominate the PCA. If temperatures were in Fahrenheit for some cities and Celsius for others, the results would be meaningless.

2. **Equal importance**: Standardization gives each variable equal initial importance before PCA determines their true contribution.

3. **Interpretability**: After standardization, loadings represent correlations between variables and principal components.

**When to Skip Standardization:**
- When all variables are already on the same scale AND you want variance differences to matter
- Example: If all 12 months are in °C and summer month variability is genuinely more important, you might analyze raw data

In [43]:
# Standardize the data (crucial for PCA when features are on different scales)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print("After standardization:")
print(f"Mean per feature: {X.mean(axis=0).round(10)}")
print(f"Std per feature: {X.std(axis=0).round(2)}")

After standardization:
Mean per feature: [ 0.  0. -0. -0. -0. -0.  0.  0.  0. -0.  0.  0.]
Std per feature: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Cell 7: Manual PCA Implementation (Understanding the Mathematics)

This cell implements PCA **from scratch** to understand the mathematical foundations. The algorithm consists of five key steps:

**Step 1: Compute the Covariance Matrix**
$$\mathbf{C} = \frac{1}{n-1}\mathbf{X}^T\mathbf{X}$$
where $\mathbf{X}$ is the centered (mean-subtracted) data matrix. The covariance matrix is $p \times p$ and symmetric.

**Step 2: Eigendecomposition**
We solve the eigenvalue problem:
$$\mathbf{C}\mathbf{v}_k = \lambda_k\mathbf{v}_k$$
- $\lambda_k$ = eigenvalue (variance captured by component $k$)
- $\mathbf{v}_k$ = eigenvector (the loading vector for component $k$)

**Step 3: Sort by Eigenvalue**
Eigenvalues are sorted in descending order: $\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_p$. The first eigenvalue corresponds to the direction of maximum variance.

**Step 4: Select Top $k$ Components**
We typically keep only the first $k$ components that capture most of the variance, discarding the rest.

**Step 5: Project Data onto Principal Components**
$$\mathbf{T} = \mathbf{X}\mathbf{P}$$
where $\mathbf{P}$ is the matrix of loadings (eigenvectors) and $\mathbf{T}$ contains the **scores** (coordinates in PC space).

**Key Outputs:**
- **Scores**: New coordinates for each observation in the rotated coordinate system
- **Loadings**: The contribution of each original variable to each PC
- **Eigenvalues**: The variance explained by each component
- **Explained variance ratio**: Percentage of total variance captured by each component

In [44]:
# Manual PCA implementation to understand the mathematics
def manual_pca(X, n_components=None):
    """Implement PCA from scratch to understand the math."""
    n_samples, n_features = X.shape
    if n_components is None:
        n_components = min(n_samples, n_features)
    
    # Step 1: Compute covariance matrix
    cov_matrix = np.cov(X.T)
    
    # Step 2: Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # Step 3: Sort by eigenvalue (descending)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Step 4: Select top k components
    components = eigenvectors[:, :n_components].T  # shape: (k, p)
    eigenvalues = eigenvalues[:n_components]
    
    # Step 5: Project data
    scores = X @ components.T  # shape: (n, k)
    
    return {
        'scores': scores,
        'loadings': components,
        'eigenvalues': eigenvalues,
        'explained_variance_ratio': eigenvalues / eigenvalues.sum(),
        'cov_matrix': cov_matrix
    }

# Run manual PCA
pca_manual = manual_pca(X)
print("Eigenvalues (variance captured by each PC):")
for i, (ev, evr) in enumerate(zip(pca_manual['eigenvalues'], 
                                    pca_manual['explained_variance_ratio'])):
    print(f"  PC{i+1}: {ev:.4f} ({evr*100:.2f}%)")

Eigenvalues (variance captured by each PC):
  PC1: 10.5212 (85.02%)
  PC2: 1.2603 (10.18%)
  PC3: 0.2670 (2.16%)
  PC4: 0.1088 (0.88%)
  PC5: 0.0732 (0.59%)
  PC6: 0.0517 (0.42%)
  PC7: 0.0309 (0.25%)
  PC8: 0.0179 (0.14%)
  PC9: 0.0160 (0.13%)
  PC10: 0.0142 (0.12%)
  PC11: 0.0088 (0.07%)
  PC12: 0.0050 (0.04%)


### Cell 8: Validation Against Scikit-Learn's PCA

This cell validates our manual implementation by comparing results with scikit-learn's optimized PCA.

**Why Compare?**
- Ensures our understanding of the mathematics is correct
- Identifies any numerical precision differences
- Shows that our manual implementation and sklearn give equivalent results

**Note on Variance Calculation:**
You may notice small numerical differences between manual and sklearn eigenvalues. This is because:
- Our manual implementation uses $n-1$ in the covariance denominator (sample covariance)
- sklearn also uses $n-1$ but may have slightly different numerical precision
- These differences are negligible and don't affect interpretation

**Going Forward:**
After validation, we use sklearn's `PCA` class for the rest of the analysis because:
1. It's numerically stable and optimized
2. It handles edge cases properly
3. It provides convenient methods like `transform()` and `inverse_transform()`

In [45]:
# Compare with sklearn's PCA
pca = PCA()
scores = pca.fit_transform(X)

print("\nSklearn PCA validation:")
print(f"Manual eigenvalues:  {pca_manual['eigenvalues'][:3].round(4)}")
print(f"Sklearn eigenvalues: {pca.explained_variance_[:3].round(4)}")
print("\n✓ Results match (sklearn uses n-1 denominator for variance)")


Sklearn PCA validation:
Manual eigenvalues:  [10.5212  1.2603  0.267 ]
Sklearn eigenvalues: [10.5212  1.2603  0.267 ]

✓ Results match (sklearn uses n-1 denominator for variance)


### Cell 9: Storing Key PCA Results for Later Use

This cell extracts and stores the essential PCA outputs in convenient variable names for subsequent analysis.

**Variables Stored:**

1. **`loadings`** (shape: $p \times k$):
   - The eigenvectors arranged as columns
   - `loadings[j, k]` = contribution of variable $j$ to component $k$
   - Note: sklearn stores components as rows, so we transpose

2. **`eigenvalues`** (length: $k$):
   - The variance captured by each principal component
   - Larger eigenvalues = more important components

3. **`explained_var`** (length: $k$):
   - Proportion of total variance explained by each component
   - $\text{explained\_var}_k = \frac{\lambda_k}{\sum_{i=1}^{p}\lambda_i}$

4. **`cumulative_var`** (length: $k$):
   - Running sum of explained variance ratios
   - Used to determine how many components capture a target percentage (e.g., 90%) of variance

**Why Store These?**
These variables are fundamental to all PCA interpretations and will be used repeatedly throughout the notebook for:
- Component selection criteria
- Loading analysis and interpretation
- Biplot construction
- Variance decomposition

In [46]:
# Store key PCA results for later use
loadings = pca.components_.T  # shape: (features, components)
eigenvalues = pca.explained_variance_
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

---
## Part 3: Variance Analysis & Scree Plots

### Cell 10: Scree Plot with Multiple Decision Criteria

This cell creates a comprehensive visualization for **selecting the optimal number of principal components** using multiple established criteria.

**Left Panel: Scree Plot**
A scree plot shows eigenvalues vs. component number. The name comes from the geological term "scree" (rocky debris at the base of a cliff).

**Three Decision Criteria Shown:**

1. **Kaiser Criterion** (red dashed line at λ=1):
   - Keep components with eigenvalue > 1
   - Rationale: For standardized data, an average variable contributes variance = 1. Components with λ < 1 explain less variance than a single original variable.
   - Pros: Simple, widely used
   - Cons: Can over/under-extract in some cases

2. **Parallel Analysis** (green dotted line):
   - Compare eigenvalues to those from random data of the same dimensions
   - Keep components where real eigenvalue > 95th percentile of random eigenvalues
   - Rationale: Only keep components that explain more variance than expected by chance
   - Pros: More rigorous than Kaiser
   - Cons: Computationally intensive (requires simulation)

3. **Elbow Method** (visual inspection):
   - Look for the "elbow" where eigenvalues level off
   - Components after the elbow contribute little additional variance

**Right Panel: Cumulative Variance**
Shows the running total of variance explained. Common thresholds:
- 80%: Often sufficient for exploratory analysis
- 90%: Standard threshold for most applications
- 95%: Conservative threshold

In [47]:
# Comprehensive scree plot with multiple decision criteria
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Scree Plot with Decision Criteria',
                                   'Cumulative Variance Explained'))

# Left plot: Scree plot
x_vals = list(range(1, len(eigenvalues) + 1))

# Eigenvalues as bars
fig.add_trace(go.Bar(x=x_vals, y=eigenvalues, name='Eigenvalue',
                     marker_color='steelblue', opacity=0.7), row=1, col=1)

# Kaiser criterion (eigenvalue = 1 for standardized data)
fig.add_hline(y=1, line_dash='dash', line_color='red', row=1, col=1,
              annotation_text='Kaiser Criterion (λ=1)', annotation_position='right')

# Parallel analysis simulation (random data eigenvalues)
np.random.seed(42)
random_eigenvalues = []
for _ in range(100):
    random_data = np.random.randn(X.shape[0], X.shape[1])
    random_pca = PCA().fit(random_data)
    random_eigenvalues.append(random_pca.explained_variance_)
parallel_line = np.percentile(random_eigenvalues, 95, axis=0)

fig.add_trace(go.Scatter(x=x_vals, y=parallel_line, mode='lines+markers',
                         name='95th Percentile (Random)', line=dict(dash='dot', color='green')),
              row=1, col=1)

# Right plot: Cumulative variance
fig.add_trace(go.Scatter(x=x_vals, y=cumulative_var * 100, mode='lines+markers',
                         name='Cumulative %', line=dict(color='purple', width=3),
                         marker=dict(size=10)), row=1, col=2)

# Add threshold lines
for threshold in [80, 90, 95]:
    fig.add_hline(y=threshold, line_dash='dash', line_color='gray', 
                  row=1, col=2, annotation_text=f'{threshold}%')

fig.update_layout(height=450, width=1100, showlegend=True)
fig.update_xaxes(title_text='Principal Component', row=1, col=1)
fig.update_xaxes(title_text='Principal Component', row=1, col=2)
fig.update_yaxes(title_text='Eigenvalue', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Variance (%)', row=1, col=2)
fig.show()

### Cell 11: Variance Explained Summary Table

This cell creates a detailed DataFrame summarizing the variance explained by each principal component.

**Columns Explained:**

| Column | Description |
|--------|-------------|
| `PC` | Principal component identifier (PC1, PC2, ...) |
| `Eigenvalue` | Variance captured (λ) - the actual eigenvalue from decomposition |
| `Variance %` | Individual contribution as percentage: $100 \times \frac{\lambda_k}{\sum\lambda}$ |
| `Cumulative %` | Running total of variance explained |
| `Keep (Kaiser)` | Boolean: True if eigenvalue > 1 |
| `Keep (Parallel)` | Boolean: True if eigenvalue exceeds 95th percentile from random data |

**How to Read This Table:**
- PC1 typically captures the majority of variance (often 70-90% in climate data)
- Each subsequent PC captures decreasing amounts
- The cumulative column helps you decide how many components to retain
- The Kaiser and Parallel columns provide rule-based recommendations

**Important Note:**
The total variance explained by all components always equals 100% (the total variance in the standardized data). PCA doesn't create or destroy variance—it merely rotates and prioritizes the axes.

In [48]:
# Detailed variance table
variance_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(eigenvalues))],
    'Eigenvalue': eigenvalues,
    'Variance %': explained_var * 100,
    'Cumulative %': cumulative_var * 100,
    'Keep (Kaiser)': eigenvalues > 1,
    'Keep (Parallel)': eigenvalues > parallel_line
})
variance_df.round(3)

,PC,Eigenvalue,Variance %,Cumulative %,Keep (Kaiser),Keep (Parallel)
0,PC1,10.521,85.020,85.020,True,True
1,PC2,1.260,10.184,95.204,True,False
2,PC3,0.267,2.158,97.361,False,False
3,PC4,0.109,0.879,98.241,False,False
4,PC5,0.073,0.591,98.832,False,False
5,PC6,0.052,0.418,99.250,False,False
6,PC7,0.031,0.250,99.499,False,False
7,PC8,0.018,0.145,99.644,False,False
8,PC9,0.016,0.129,99.774,False,False
9,PC10,0.014,0.115,99.889,False,False


### Cell 12: Optimal Component Selection Summary

This cell computes and displays the recommended number of components using three different criteria.

**Criteria Comparison:**

1. **Kaiser Criterion** (`kaiser_k`):
   - Counts components where eigenvalue > 1
   - Simple threshold based on average variable contribution
   - Works well for standardized data

2. **Parallel Analysis** (`parallel_k`):
   - Compares to simulated random data
   - More conservative than Kaiser in most cases
   - Considered the gold standard in psychology research

3. **90% Variance Threshold** (`var90_k`):
   - Finds the minimum number of components to reach 90% cumulative variance
   - Goal-oriented approach: "How many components do I need to retain 90% of information?"

**Making the Final Decision:**
The recommendation uses `max(kaiser_k, parallel_k)` because:
- We want to avoid under-extraction (missing important patterns)
- Both criteria have theoretical justification
- In practice, having one extra component rarely hurts analysis

**Context Matters:**
Your final choice should also consider:
- Domain knowledge (do the components have meaningful interpretations?)
- Practical constraints (how many components can you reasonably visualize/interpret?)
- Purpose (exploratory vs. predictive modeling)

In [49]:
# Determine optimal number of components
kaiser_k = (eigenvalues > 1).sum()
parallel_k = (eigenvalues > parallel_line).sum()
var90_k = np.argmax(cumulative_var >= 0.90) + 1

print("Component Selection Criteria:")
print(f"  Kaiser criterion (λ > 1):        {kaiser_k} components")
print(f"  Parallel analysis (95%):         {parallel_k} components")
print(f"  90% variance explained:          {var90_k} components")
print(f"\n→ Recommended: {max(kaiser_k, parallel_k)} components")

Component Selection Criteria:
  Kaiser criterion (λ > 1):        2 components
  Parallel analysis (95%):         1 components
  90% variance explained:          2 components

→ Recommended: 2 components


---
## Part 4: Component Interpretation - Loadings Analysis

### Cell 13: Creating the Loadings DataFrame

This cell creates a structured DataFrame containing the **loading values** for the first 4 principal components.

**What are Loadings?**
Loadings represent the **coefficients** that define each principal component as a linear combination of original variables:
$$PC_k = \sum_{j=1}^{p} l_{jk} \cdot Z_j$$

where $l_{jk}$ is the loading of variable $j$ on component $k$, and $Z_j$ is the standardized variable.

**Interpreting Loading Values:**
- **Magnitude** (|loading|): How strongly the variable contributes to the component
  - |loading| > 0.4: Strong contribution
  - |loading| 0.2-0.4: Moderate contribution
  - |loading| < 0.2: Weak contribution

- **Sign** (+/-): Direction of the relationship
  - Same sign = variables move together along this component
  - Opposite signs = variables move in opposite directions

**For Standardized Data:**
When PCA is performed on standardized data (as we did), loadings can be interpreted as **correlations** between the original variables and the principal components:
$$\text{correlation}(X_j, PC_k) \approx l_{jk} \times \sqrt{\lambda_k}$$

**DataFrame Structure:**
- Rows: Original variables (months)
- Columns: Principal components (PC1-PC4)

In [50]:
# Create loadings DataFrame
loadings_df = pd.DataFrame(
    loadings[:, :4],
    index=month_cols,
    columns=['PC1', 'PC2', 'PC3', 'PC4']
)

print("Principal Component Loadings (first 4 PCs):")
print(loadings_df.round(3).to_string())

Principal Component Loadings (first 4 PCs):
       PC1    PC2    PC3    PC4
Jan  0.276 -0.398  0.174 -0.046
Feb  0.284 -0.359  0.092 -0.230
Mar  0.295 -0.241 -0.120 -0.444
Apr  0.300  0.026 -0.444 -0.398
May  0.283  0.326 -0.336  0.111
Jun  0.273  0.338  0.562 -0.254
Jul  0.269  0.405  0.420 -0.121
Aug  0.280  0.335 -0.181  0.349
Sep  0.306  0.118 -0.199  0.052
Oct  0.310 -0.006 -0.107  0.185
Nov  0.303 -0.175 -0.003  0.286
Dec  0.280 -0.336  0.246  0.507


### Cell 14: Loadings Heatmap Visualization

This cell creates a **heatmap** to visualize the loading matrix, making patterns easier to identify at a glance.

**Color Encoding:**
- **Red/positive**: Variable increases when component increases
- **Blue/negative**: Variable decreases when component increases  
- **White/neutral**: Variable has little association with component

**Visual Pattern Recognition:**

1. **PC1 Pattern**: Look for uniform coloring across all months
   - If all months are the same color → PC1 captures overall temperature level
   - This is the "size" or "average" component

2. **PC2 Pattern**: Look for contrasting patterns
   - If summer months are red and winter months are blue (or vice versa) → PC2 captures seasonality
   - This is the "shape" or "contrast" component

3. **Higher PCs**: Look for more complex patterns
   - PC3, PC4 may capture shoulder seasons (spring/fall) or other subtleties
   - These typically explain much less variance

**Reading the Heatmap:**
- Rows = Principal components (what you're describing)
- Columns = Original variables (months)
- Cell value = How much that month contributes to that PC

In [51]:
# Visualize loadings as a heatmap
fig = px.imshow(loadings_df.T, text_auto='.2f', aspect='auto',
                color_continuous_scale='RdBu_r', zmin=-0.5, zmax=0.5,
                labels=dict(x='Month', y='Component', color='Loading'))
fig.update_layout(title='PCA Loadings Heatmap', width=900, height=350)
fig.show()

### Cell 15: Loading Profiles as Bar Charts

This cell creates **bar charts** for each principal component's loadings, displayed as separate subplots.

**Why Bar Charts?**
While heatmaps show the overall pattern, bar charts allow:
- Precise comparison of loading magnitudes
- Easy identification of which months load positively vs. negatively
- Clear visualization of the loading "profile" for each component

**Interpreting Each Component:**

**PC1 (≈85% variance):**
- If all bars are roughly equal height and same direction → "Average/Level" component
- A city with high PC1 score is warmer across ALL months

**PC2 (≈10% variance):**
- If summer months positive and winter months negative → "Seasonality/Contrast" component
- High PC2 = large summer-winter temperature difference (continental climate)
- Low PC2 = small seasonal variation (maritime/oceanic climate)

**PC3, PC4:**
- Typically capture subtler patterns like:
  - Spring vs. fall asymmetry
  - Specific monthly anomalies
- Usually explain <5% variance each

**The Zero Line:**
The dashed horizontal line at y=0 helps identify sign changes. Sign flips indicate which variables move in opposite directions along that component.

In [52]:
# Loadings as line profiles (easier to interpret seasonal patterns)
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f'PC1 ({explained_var[0]*100:.1f}%)', f'PC2 ({explained_var[1]*100:.1f}%)',
    f'PC3 ({explained_var[2]*100:.1f}%)', f'PC4 ({explained_var[3]*100:.1f}%)'
])

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
for i, (col, c) in enumerate(zip(['PC1', 'PC2', 'PC3', 'PC4'], colors)):
    row, col_idx = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(go.Bar(x=month_cols, y=loadings_df[col], marker_color=c, name=col),
                  row=row, col=col_idx)
    fig.add_hline(y=0, line_dash='dash', line_color='black', row=row, col=col_idx)

fig.update_layout(height=500, width=1000, showlegend=False,
                  title_text='Loading Profiles by Component')
fig.show()

### Interpretation of Principal Components

**PC1 (≈85% variance):** All months load positively with roughly equal weights.
- This represents the **overall temperature level** or "average warmth" of a city
- Cities with high PC1 scores are warmer year-round (Mediterranean cities)
- Cities with low PC1 scores are colder year-round (Northern cities)

**PC2 (≈10% variance):** Summer months positive, winter months negative.
- This represents **seasonality** or temperature amplitude
- High PC2 = Continental climate (hot summers, cold winters)
- Low PC2 = Oceanic/Maritime climate (mild year-round)

### Cell 16: The Biplot - Combining Scores and Loadings

This cell creates a **biplot**, one of the most powerful PCA visualization tools that displays both observations (scores) and variables (loadings) in the same plot.

**What is a Biplot?**
A biplot superimposes two plots:
1. **Scatter plot of scores**: Each point is an observation (city) plotted at its PC coordinates
2. **Arrows for loadings**: Each arrow represents a variable (month), pointing in the direction of maximum correlation with the PCs

**How to Read a Biplot:**

**Interpreting Points (Cities):**
- Points close together have similar temperature profiles
- Points far from origin are more extreme (higher scores on one or both axes)
- The x-coordinate is the PC1 score; y-coordinate is the PC2 score

**Interpreting Arrows (Months):**
- Arrow direction: Shows the "direction" of that variable in PC space
- Arrow length: Proportional to how well the variable is represented in 2D (longer = better represented)
- Arrows pointing same direction: Variables are positively correlated
- Arrows pointing opposite directions: Variables are negatively correlated
- Arrows at 90°: Variables are uncorrelated

**City-Variable Relationships:**
- A city located in the direction of an arrow has high values for that variable
- A city opposite to an arrow has low values for that variable
- Distance from origin to city projection on arrow direction indicates the value

**The Scaling Factor:**
Loadings are scaled (multiplied by a constant) for visual clarity. This doesn't change interpretation—only the visual length of arrows.

In [53]:
# Biplot: Scores and Loadings together
def create_biplot(scores, loadings, features, labels, pc_x=0, pc_y=1, scale=3):
    """Create an interactive biplot."""
    fig = go.Figure()
    
    # Plot scores (observations)
    fig.add_trace(go.Scatter(
        x=scores[:, pc_x], y=scores[:, pc_y],
        mode='markers+text', text=labels,
        textposition='top center', textfont=dict(size=9),
        marker=dict(size=10, color='steelblue', opacity=0.7),
        name='Cities'
    ))
    
    # Plot loadings (variables) as arrows
    for i, feature in enumerate(features):
        fig.add_annotation(
            x=loadings[i, pc_x] * scale, y=loadings[i, pc_y] * scale,
            ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
            showarrow=True, arrowhead=2, arrowsize=1.5, arrowwidth=2,
            arrowcolor='red'
        )
        fig.add_annotation(
            x=loadings[i, pc_x] * scale * 1.1, y=loadings[i, pc_y] * scale * 1.1,
            text=feature, showarrow=False, font=dict(color='red', size=11)
        )
    
    fig.update_layout(
        title=f'PCA Biplot (PC{pc_x+1} vs PC{pc_y+1})',
        xaxis_title=f'PC{pc_x+1} ({explained_var[pc_x]*100:.1f}%)',
        yaxis_title=f'PC{pc_y+1} ({explained_var[pc_y]*100:.1f}%)',
        width=900, height=700,
        xaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor='gray'),
        yaxis=dict(zeroline=True, zerolinewidth=1, zerolinecolor='gray')
    )
    return fig

fig = create_biplot(scores, loadings, month_cols, cities)
fig.show()

---
## Part 5: Discovering Causes of Variation

### Cell 17: Creating a Results DataFrame with External Variables

This cell combines PCA results with **external metadata** (latitude, calculated statistics) to investigate what real-world factors the principal components represent.

**Variables in the DataFrame:**

| Variable | Source | Description |
|----------|--------|-------------|
| `City` | Original data | City identifier |
| `Latitude` | Original data | Geographic latitude (°N) |
| `PC1, PC2, PC3` | PCA scores | Coordinates in PC space |
| `Avg_Temp` | Calculated | Mean of 12 monthly temperatures |
| `Temp_Range` | Calculated | Max - Min monthly temperature (seasonality indicator) |

**Statistical Analysis:**
We calculate Pearson correlations between PC scores and external variables to discover what each component represents:

1. **PC1 vs. Latitude**: Expect strong negative correlation (northern cities are colder)
2. **PC1 vs. Avg_Temp**: Expect strong positive correlation (PC1 ≈ overall warmth)
3. **PC2 vs. Temp_Range**: Expect strong correlation (PC2 ≈ seasonality)

**Why This Matters:**
This analysis bridges the gap between abstract statistical components and **interpretable real-world factors**. We're essentially asking: "What does PC1 actually measure in the real world?"

**p-values:**
Small p-values (< 0.05) indicate statistically significant correlations, suggesting the relationship is unlikely to be due to chance.

In [54]:
# Relate PC scores to external variables (Latitude)
results_df = pd.DataFrame({
    'City': cities,
    'Latitude': latitudes,
    'PC1': scores[:, 0],
    'PC2': scores[:, 1],
    'PC3': scores[:, 2],
    'Avg_Temp': X_raw.mean(axis=1),
    'Temp_Range': X_raw.max(axis=1) - X_raw.min(axis=1)
})

# Calculate correlations
print("Correlation with Latitude:")
for pc in ['PC1', 'PC2', 'PC3']:
    r, p = stats.pearsonr(results_df['Latitude'], results_df[pc])
    print(f"  {pc}: r = {r:.3f}, p = {p:.4f}")

print("\nCorrelation with Average Temperature:")
for pc in ['PC1', 'PC2', 'PC3']:
    r, p = stats.pearsonr(results_df['Avg_Temp'], results_df[pc])
    print(f"  {pc}: r = {r:.3f}, p = {p:.4f}")

print("\nCorrelation with Temperature Range (Seasonality):")
for pc in ['PC1', 'PC2', 'PC3']:
    r, p = stats.pearsonr(results_df['Temp_Range'], results_df[pc])
    print(f"  {pc}: r = {r:.3f}, p = {p:.4f}")

Correlation with Latitude:
  PC1: r = -0.877, p = 0.0000
  PC2: r = -0.195, p = 0.2776
  PC3: r = 0.211, p = 0.2377

Correlation with Average Temperature:
  PC1: r = 0.998, p = 0.0000
  PC2: r = -0.061, p = 0.7374
  PC3: r = 0.012, p = 0.9482

Correlation with Temperature Range (Seasonality):
  PC1: r = -0.457, p = 0.0075
  PC2: r = 0.859, p = 0.0000
  PC3: r = -0.130, p = 0.4698


### Cell 18: Visualizing PC-Variable Relationships with Scatter Plots

This cell creates scatter plots with regression lines to visually confirm the correlations computed in the previous cell.

**Left Panel: PC1 vs. Latitude**
- Each point is a city
- X-axis: Geographic latitude (°N)
- Y-axis: PC1 score
- Color: Average temperature (red=warm, blue=cold)
- Red dashed line: Linear regression fit

**Expected Pattern:**
Strong negative slope—as latitude increases (going north), PC1 decreases (colder overall). This confirms PC1 captures the north-south temperature gradient.

**Right Panel: PC2 vs. Temperature Range**
- X-axis: Temperature range (max month - min month)
- Y-axis: PC2 score  
- Color: Latitude (yellow=south, purple=north)

**Expected Pattern:**
Strong positive slope—cities with larger temperature swings (continental climate) have higher PC2 scores.

**Regression Statistics:**
- `r` = Pearson correlation coefficient (shown on line labels)
- Values close to ±1 indicate strong linear relationships
- These plots provide visual validation of the numerical correlations

**Interpretation Framework:**
After this analysis, we can confidently state:
- **PC1 ≈ Latitude effect ≈ Overall temperature level**
- **PC2 ≈ Continentality ≈ Seasonal amplitude**

In [55]:
# Visualize PC1 vs Latitude relationship
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'PC1 vs Latitude (Overall Temperature)', 
    'PC2 vs Temperature Range (Seasonality)'
])

# PC1 vs Latitude
fig.add_trace(go.Scatter(
    x=results_df['Latitude'], y=results_df['PC1'],
    mode='markers+text', text=results_df['City'],
    textposition='top center', textfont=dict(size=8),
    marker=dict(size=12, color=results_df['Avg_Temp'], 
                colorscale='RdYlBu_r', showscale=True,
                colorbar=dict(title='Avg Temp', x=0.45)),
    showlegend=False
), row=1, col=1)

# Add regression line
slope1, intercept1, r1, p1, se1 = stats.linregress(results_df['Latitude'], results_df['PC1'])
x_line = np.linspace(results_df['Latitude'].min(), results_df['Latitude'].max(), 100)
fig.add_trace(go.Scatter(x=x_line, y=slope1*x_line + intercept1,
                         mode='lines', line=dict(dash='dash', color='red'),
                         name=f'r={r1:.2f}'), row=1, col=1)

# PC2 vs Temperature Range
fig.add_trace(go.Scatter(
    x=results_df['Temp_Range'], y=results_df['PC2'],
    mode='markers+text', text=results_df['City'],
    textposition='top center', textfont=dict(size=8),
    marker=dict(size=12, color=results_df['Latitude'],
                colorscale='Viridis', showscale=True,
                colorbar=dict(title='Latitude', x=1.02)),
    showlegend=False
), row=1, col=2)

slope2, intercept2, r2, p2, se2 = stats.linregress(results_df['Temp_Range'], results_df['PC2'])
x_line2 = np.linspace(results_df['Temp_Range'].min(), results_df['Temp_Range'].max(), 100)
fig.add_trace(go.Scatter(x=x_line2, y=slope2*x_line2 + intercept2,
                         mode='lines', line=dict(dash='dash', color='red'),
                         name=f'r={r2:.2f}'), row=1, col=2)

fig.update_layout(height=500, width=1100)
fig.update_xaxes(title_text='Latitude (°N)', row=1, col=1)
fig.update_xaxes(title_text='Temperature Range (°C)', row=1, col=2)
fig.update_yaxes(title_text='PC1 Score', row=1, col=1)
fig.update_yaxes(title_text='PC2 Score', row=1, col=2)
fig.show()

### Key Discoveries

**PC1 = Geographic/Climatic Zone** 
- Strong negative correlation with latitude: northern cities score lower
- Captures the fundamental north-south temperature gradient across Europe

**PC2 = Continentality**
- Strong positive correlation with temperature range
- Distinguishes oceanic climates (coastal, mild) from continental climates (inland, extreme)

---
## Part 6: Anomaly Detection

### Cell 19: PCA Diagnostic Function for Anomaly Detection

This cell defines a comprehensive function to calculate **three key PCA diagnostics** used for anomaly/outlier detection:

**1. Hotelling's T² Statistic**
$$T^2_i = \sum_{k=1}^{K} \frac{t_{ik}^2}{\lambda_k}$$

Where $t_{ik}$ is the score of observation $i$ on component $k$, and $\lambda_k$ is the eigenvalue.

- **Interpretation**: Measures how far an observation is from the center in the **model space** (the space spanned by retained PCs)
- **High T²**: The observation follows the model's patterns but is extreme (e.g., an extremely warm city that's warm in the normal seasonal way)

**2. Q-Residual (SPE - Squared Prediction Error)**
$$Q_i = \sum_{j=1}^{p} e_{ij}^2 = \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$$

Where $\hat{\mathbf{x}}_i$ is the reconstruction of observation $i$ using $K$ components.

- **Interpretation**: Measures variation **outside the model** (in the residual space)
- **High Q**: The observation has patterns not captured by the model (e.g., a city with unusual seasonal timing)

**3. Leverage**
$$h_i = \sum_{k=1}^{K} \frac{t_{ik}^2}{n \cdot \lambda_k} + \frac{1}{n}$$

- **Interpretation**: How much influence observation $i$ has on the PCA model
- **High leverage**: Removing this observation would change the PCs significantly

**Control Limits:**
- **T² limit**: Based on F-distribution at significance level α
- **Q limit**: Based on weighted chi-squared approximation

These diagnostics are fundamental to **multivariate statistical process control (MSPC)**.

In [56]:
def calculate_pca_diagnostics(X, n_components=2):
    """
    Calculate comprehensive PCA diagnostics for anomaly detection.
    
    Returns:
    - T2: Hotelling's T² (variation in model space)
    - Q: Q-residual / SPE (variation outside model space)
    - leverage: Observation leverage (influence on model)
    """
    n, p = X.shape
    k = n_components
    
    # Fit PCA
    pca_model = PCA(n_components=k)
    T = pca_model.fit_transform(X)  # Scores
    P = pca_model.components_.T     # Loadings (p x k)
    eigenvalues = pca_model.explained_variance_
    
    # Hotelling's T² (Mahalanobis distance in PC space)
    T2 = np.sum((T ** 2) / eigenvalues, axis=1)
    
    # Q-residual (Squared Prediction Error / SPE)
    X_reconstructed = T @ P.T
    residuals = X - X_reconstructed
    Q = np.sum(residuals ** 2, axis=1)
    
    # Leverage (diagonal of hat matrix)
    leverage = np.sum((T / np.sqrt(eigenvalues)) ** 2, axis=1) / n + 1/n
    
    # Control limits
    alpha = 0.05
    
    # T² limit (F-distribution)
    T2_limit = k * (n - 1) / (n - k) * stats.f.ppf(1 - alpha, k, n - k)
    
    # Q limit (approximation using chi-squared)
    residual_eigenvalues = pca_model.explained_variance_[k:] if k < min(n, p) else [0]
    theta1 = np.sum(pca_model.singular_values_[k:] ** 2) if k < len(pca_model.singular_values_) else np.var(Q)
    theta2 = np.sum(pca_model.singular_values_[k:] ** 4) if k < len(pca_model.singular_values_) else np.var(Q)**2
    theta3 = np.sum(pca_model.singular_values_[k:] ** 6) if k < len(pca_model.singular_values_) else np.var(Q)**3
    
    if theta1 > 0:
        h0 = 1 - 2 * theta1 * theta3 / (3 * theta2 ** 2)
        h0 = max(h0, 0.001)  # Ensure positive
        ca = stats.norm.ppf(1 - alpha)
        Q_limit = theta1 * (ca * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 
                           1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)
    else:
        Q_limit = np.percentile(Q, (1 - alpha) * 100)
    
    return {
        'T2': T2, 'Q': Q, 'leverage': leverage,
        'T2_limit': T2_limit, 'Q_limit': Q_limit,
        'scores': T, 'loadings': P, 'eigenvalues': eigenvalues,
        'reconstruction': X_reconstructed, 'residuals': residuals
    }

# Calculate diagnostics with 2 components
diag = calculate_pca_diagnostics(X, n_components=2)

results_df['T2'] = diag['T2']
results_df['Q'] = diag['Q']
results_df['Leverage'] = diag['leverage']

### Cell 20: The T² vs. Q Plot (Residual Outlier Map)

This cell creates the **T² vs. Q plot**, a powerful diagnostic visualization that divides observations into four categories based on their anomaly type.

**Four Quadrants Interpretation:**

| Quadrant | T² | Q | Meaning |
|----------|----|----|---------|
| **Bottom-left** | Low | Low | Normal—observation fits the model well |
| **Bottom-right** | High | Low | Extreme but consistent—follows the model's patterns intensely |
| **Top-left** | Low | High | New behavior—has patterns not captured by the model |
| **Top-right** | High | High | True outlier—extreme AND unusual pattern |

**Color Coding in the Plot:**
- 🔵 **Blue**: Normal observations (both T² and Q within limits)
- 🟠 **Orange**: High T² only (extreme in model space)
- 🟣 **Purple**: High Q only (doesn't fit the model structure)
- 🔴 **Red**: Both high (extreme outlier)

**Control Limits (Dashed Lines):**
- **Vertical red line**: T² limit based on F-distribution
- **Horizontal red line**: Q limit based on chi-squared approximation

**Why Use Both T² and Q?**
- T² alone misses observations with novel patterns (they might have low scores but high residuals)
- Q alone misses extreme but consistent observations
- Together, they provide complete anomaly coverage

**Practical Example:**
- City with high T², low Q: Very warm city, but seasonal pattern is typical
- City with low T², high Q: Average temperature, but unusual seasonal pattern (e.g., bimodal)

In [57]:
# T² vs Q plot (Residual Outlier Map)
fig = go.Figure()

# Color points by anomaly type
colors = []
for t2, q in zip(diag['T2'], diag['Q']):
    if t2 > diag['T2_limit'] and q > diag['Q_limit']:
        colors.append('red')     # Both high: extreme outlier
    elif t2 > diag['T2_limit']:
        colors.append('orange')  # High T²: unusual in model space
    elif q > diag['Q_limit']:
        colors.append('purple')  # High Q: doesn't fit model
    else:
        colors.append('steelblue')  # Normal

fig.add_trace(go.Scatter(
    x=diag['T2'], y=diag['Q'],
    mode='markers+text', text=cities,
    textposition='top center', textfont=dict(size=9),
    marker=dict(size=12, color=colors, line=dict(width=1, color='black')),
    hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
))

# Add control limits
fig.add_vline(x=diag['T2_limit'], line_dash='dash', line_color='red',
              annotation_text=f"T² limit ({diag['T2_limit']:.2f})")
fig.add_hline(y=diag['Q_limit'], line_dash='dash', line_color='red',
              annotation_text=f"Q limit ({diag['Q_limit']:.2f})")

# Add quadrant labels
fig.add_annotation(x=diag['T2_limit']*0.3, y=diag['Q_limit']*0.3,
                   text='Normal', showarrow=False, font=dict(size=14, color='green'))
fig.add_annotation(x=diag['T2'].max()*0.8, y=diag['Q_limit']*0.3,
                   text='Extreme in<br>model space', showarrow=False, font=dict(size=12, color='orange'))
fig.add_annotation(x=diag['T2_limit']*0.3, y=diag['Q'].max()*0.8,
                   text='New behavior<br>(not in model)', showarrow=False, font=dict(size=12, color='purple'))

fig.update_layout(
    title="PCA Residual Outlier Map (T² vs Q)",
    xaxis_title="Hotelling's T² (variation in model)",
    yaxis_title="Q Residual (variation outside model)",
    width=900, height=700
)
fig.show()

### Cell 21: Identifying and Reporting Anomalies

This cell identifies which cities exceed the T² and/or Q control limits and provides interpretation guidelines.

**Output Interpretation:**

**T² Outliers** (extreme within model):
- These cities have unusual temperature levels but follow the expected seasonal pattern
- Example: A city that is exceptionally warm or cold, but has normal summer-winter variation
- Actionable insight: Worth investigating why this city is so extreme on the dominant patterns

**Q Outliers** (don't fit the model):
- These cities have temperature patterns not captured by the first 2 principal components
- Example: A city with an unusual monsoon season, or Mediterranean climate with winter rain effects
- Actionable insight: May need additional components or separate analysis

**Both T² and Q High:**
- Truly unusual observations that are both extreme AND have novel patterns
- These warrant the most attention in any analysis

**Why This Matters:**
1. **Quality control**: In industrial applications, high Q indicates a process behaving differently than historical norms
2. **Data cleaning**: Outliers may indicate measurement errors or data entry mistakes
3. **Discovery**: Sometimes outliers represent genuinely interesting cases worth studying further
4. **Model validation**: If many observations have high Q, the model may need more components

In [58]:
# Identify outliers and explain why
outliers_T2 = results_df[results_df['T2'] > diag['T2_limit']]['City'].tolist()
outliers_Q = results_df[results_df['Q'] > diag['Q_limit']]['City'].tolist()

print("=" * 60)
print("ANOMALY DETECTION RESULTS")
print("=" * 60)
print(f"\nT² outliers (extreme within model): {outliers_T2 if outliers_T2 else 'None'}")
print(f"Q outliers (don't fit the model): {outliers_Q if outliers_Q else 'None'}")
print("\n" + "-" * 60)
print("Interpretation:")
print("-" * 60)
print("• High T² = City follows the same patterns but more extremely")
print("• High Q = City has a unique pattern not captured by the model")
print("• Both high = Truly unusual observation")

ANOMALY DETECTION RESULTS

T² outliers (extreme within model): ['Marbella', 'Moscow']
Q outliers (don't fit the model): ['Vienna']

------------------------------------------------------------
Interpretation:
------------------------------------------------------------
• High T² = City follows the same patterns but more extremely
• High Q = City has a unique pattern not captured by the model
• Both high = Truly unusual observation


### Cell 22: Contribution Plots for Diagnosing Anomalies

This cell creates **contribution plots** that decompose T² and Q statistics by variable, answering the question: "Which months are responsible for this city being an outlier?"

**T² Contribution Plot (Left Panel):**
Shows how much each month contributes to the observation's T² statistic:
$$\text{T² contribution}_j \approx \sum_{k=1}^{K} \frac{(t_k \cdot p_{jk})^2}{\lambda_k}$$

- **Interpretation**: Which months are driving the extreme position in PC space?
- **High bar**: That month's value is contributing heavily to the city's unusual score
- **Use case**: If a city has high T² because of summer months, you know it's the summer temperatures that are extreme

**Q Contribution Plot (Right Panel):**
Shows the squared residual for each month:
$$\text{Q contribution}_j = e_j^2 = (x_j - \hat{x}_j)^2$$

- **Interpretation**: Which months have the largest reconstruction error?
- **High bar**: The model failed to predict that month's temperature accurately
- **Use case**: Identifies which specific months don't fit the general pattern

**Diagnostic Workflow:**
1. Identify outlier in T² vs. Q plot
2. Generate contribution plot for that observation
3. Identify problematic variables (months)
4. Investigate root cause (measurement error? genuine anomaly? external factor?)

**Why "Most Extreme":**
We automatically select the city with the highest combined anomaly score $(T^2/\text{limit} + Q/\text{limit})$ for detailed analysis.

In [59]:
# Contribution plot for a specific observation
def contribution_plot(X, diag, obs_idx, feature_names):
    """Show which features contribute most to T² and Q for an observation."""
    T = diag['scores']
    P = diag['loadings']
    eigenvalues = diag['eigenvalues']
    
    # T² contributions by feature
    t_contributions = np.zeros(len(feature_names))
    for j in range(len(eigenvalues)):
        t_contributions += (T[obs_idx, j] * P[:, j]) ** 2 / eigenvalues[j]
    
    # Q contributions by feature
    q_contributions = diag['residuals'][obs_idx] ** 2
    
    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        f'T² Contribution (Total: {diag["T2"][obs_idx]:.2f})',
        f'Q Contribution (Total: {diag["Q"][obs_idx]:.2f})'
    ])
    
    fig.add_trace(go.Bar(x=feature_names, y=t_contributions, marker_color='orange'), row=1, col=1)
    fig.add_trace(go.Bar(x=feature_names, y=q_contributions, marker_color='purple'), row=1, col=2)
    
    fig.update_layout(height=350, width=1000, showlegend=False,
                      title_text=f'Contribution Plot: {cities[obs_idx]}')
    return fig

# Show contribution for the most extreme city
combined_score = diag['T2'] / diag['T2_limit'] + diag['Q'] / diag['Q_limit']
most_extreme_idx = np.argmax(combined_score)

print(f"Most extreme observation: {cities[most_extreme_idx]}")
fig = contribution_plot(X, diag, most_extreme_idx, month_cols)
fig.show()

Most extreme observation: Vienna


---
## Part 7: PCA-Based Data Imputation

### Cell 23: PCA-Based Data Imputation (EM-PCA Algorithm)

This cell implements **iterative PCA imputation**, also known as the **EM-PCA algorithm**, for filling in missing values.

**The Problem:**
Real-world datasets often have missing values. Simple imputation (e.g., mean substitution) ignores correlations between variables. PCA can do better by exploiting the data structure.

**Algorithm Steps:**

1. **Initialize**: Replace missing values with column means (simple imputation)
2. **Iterate**:
   - Fit PCA on the current complete data
   - Reconstruct all data using $K$ principal components
   - Update **only the missing values** with reconstructed values
   - Keep observed values unchanged
3. **Converge**: Stop when imputed values stabilize (change < tolerance)

**Mathematical Basis:**
This is equivalent to **Expectation-Maximization (EM)** applied to a probabilistic PCA model:
- **E-step**: Estimate missing values given current PCA model
- **M-step**: Re-estimate PCA model given completed data

**Why PCA Imputation Works:**
- Exploits correlations between variables
- Missing temperatures in June can be estimated from correlated months (May, July, summer average)
- The model "knows" that cities follow predictable seasonal patterns

**Demonstration:**
We artificially remove 20 random values from the data, impute them using PCA, then compare imputed vs. actual values to assess accuracy.

**Parameters:**
- `n_components=2`: Number of PCs to use for reconstruction
- `max_iter=100`: Maximum iterations
- `tol=1e-6`: Convergence tolerance

In [60]:
def pca_imputation(X_with_missing, n_components=2, max_iter=100, tol=1e-6):
    """
    Impute missing values using iterative PCA (EM-PCA algorithm).
    
    Algorithm:
    1. Initialize missing values with column means
    2. Fit PCA and reconstruct data
    3. Update missing values with reconstructed values
    4. Repeat until convergence
    """
    X = X_with_missing.copy()
    missing_mask = np.isnan(X)
    n_missing = missing_mask.sum()
    
    if n_missing == 0:
        return X, []
    
    # Initial imputation with column means
    col_means = np.nanmean(X, axis=0)
    for j in range(X.shape[1]):
        X[missing_mask[:, j], j] = col_means[j]
    
    convergence_history = []
    prev_imputed = X[missing_mask].copy()
    
    for iteration in range(max_iter):
        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Fit PCA
        pca = PCA(n_components=n_components)
        scores = pca.fit_transform(X_scaled)
        
        # Reconstruct
        X_reconstructed_scaled = scores @ pca.components_
        X_reconstructed = scaler.inverse_transform(X_reconstructed_scaled)
        
        # Update only missing values
        X[missing_mask] = X_reconstructed[missing_mask]
        
        # Check convergence
        current_imputed = X[missing_mask]
        change = np.sqrt(np.mean((current_imputed - prev_imputed) ** 2))
        convergence_history.append(change)
        
        if change < tol:
            print(f"Converged after {iteration + 1} iterations")
            break
            
        prev_imputed = current_imputed.copy()
    
    return X, convergence_history

# Demonstrate with artificially missing data
np.random.seed(42)
X_missing = X_raw.copy()
n_to_remove = 20
missing_positions = []

for _ in range(n_to_remove):
    i, j = np.random.randint(0, X_raw.shape[0]), np.random.randint(0, X_raw.shape[1])
    missing_positions.append((i, j, X_raw[i, j]))
    X_missing[i, j] = np.nan

print(f"Introduced {n_to_remove} missing values")
print(f"Missing pattern: {np.isnan(X_missing).sum(axis=0)} per month")

Introduced 20 missing values
Missing pattern: [2 0 2 0 5 1 2 1 0 1 3 3] per month


### Cell 24: Evaluating Imputation Quality

This cell compares the **imputed values to the true (hidden) values** and calculates accuracy metrics.

**Output Table Columns:**
- **City**: Which city had the missing value
- **Month**: Which month was missing
- **Actual**: The true temperature we hid
- **Imputed**: The value predicted by PCA imputation
- **Error**: Imputed - Actual (positive = overestimate, negative = underestimate)

**Quality Metrics:**

**RMSE (Root Mean Squared Error):**
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2}$$
- Penalizes large errors more heavily
- Same units as the original data (°C)

**MAE (Mean Absolute Error):**
$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|\hat{y}_i - y_i|$$
- Average magnitude of errors
- More robust to outliers than RMSE

**Interpretation:**
- RMSE < 2°C typically indicates excellent imputation for climate data
- Compare to simple mean imputation baseline (would likely be worse)
- Larger errors may occur for unusual cities or extreme months

**Why This Validation Matters:**
In practice, we don't know the true values. This artificial test demonstrates that PCA imputation is reliable for this type of data. Similar cross-validation approaches can be used to select optimal $K$.

In [61]:
# Perform PCA imputation
X_imputed, convergence = pca_imputation(X_missing, n_components=2)

# Compare imputed vs actual values
imputation_results = []
for i, j, actual in missing_positions:
    imputed = X_imputed[i, j]
    error = imputed - actual
    imputation_results.append({
        'City': cities[i],
        'Month': month_cols[j],
        'Actual': actual,
        'Imputed': round(imputed, 1),
        'Error': round(error, 2)
    })

imputation_df = pd.DataFrame(imputation_results)
print("\nImputation Results:")
print(imputation_df.to_string(index=False))

rmse = np.sqrt(np.mean([r['Error']**2 for r in imputation_results]))
mae = np.mean([abs(r['Error']) for r in imputation_results])
print(f"\nImputation Quality: RMSE = {rmse:.2f}°C, MAE = {mae:.2f}°C")

Converged after 16 iterations

Imputation Results:
          City Month  Actual  Imputed  Error
     Ljubljana   Nov     5.6      6.1   0.54
      Bruxells   May    13.0     14.3   1.26
          Oslo   Jul    17.0     16.5  -0.52
    Copenhagen   Nov     4.0      4.2   0.20
     Stockholm   May    11.0     12.4   1.42
     Stockholm   Mar    -1.0      1.2   2.17
          Rome   May    18.0     18.9   0.88
        Ankara   Aug    22.0     22.1   0.08
        Madrid   Jun    22.2     21.8  -0.35
        Ankara   Dec     2.0      2.8   0.84
        Prague   Jan    -2.0     -2.6  -0.63
        Dublin   Oct     9.0      9.1   0.07
          Rome   Dec     9.0      7.8  -1.17
        Vienna   Jan    -1.0      2.5   3.52
    Bratislava   Nov     4.9      5.7   0.79
St. Petersberg   Dec     4.2      4.8   0.60
     Luxemborg   Mar     7.0      4.8  -2.25
        Athens   May    21.0     21.5   0.46
          Bonn   May    14.0     14.0  -0.02
      Bucarest   Jul    23.0     24.2   1.17

Imp

### Cell 25: Visualizing Imputation Convergence and Accuracy

This cell creates two diagnostic plots for the imputation process.

**Left Panel: Convergence History**
- X-axis: Iteration number
- Y-axis: Change in imputed values (RMSE between iterations)
- Shows how quickly the algorithm converges

**What to Look For:**
- Rapid initial decrease = algorithm is making progress
- Plateau = convergence achieved
- Should stabilize within 10-20 iterations for well-behaved data
- Non-convergence may indicate too many missing values or structural issues

**Right Panel: Actual vs. Imputed Scatter**
- Each point is one imputed value
- X-axis: True (actual) value
- Y-axis: Imputed value
- Red dashed line: Perfect imputation (y = x)

**What to Look For:**
- Points close to the diagonal = accurate imputation
- Systematic bias: Points consistently above/below the line
- Outliers: Individual points far from the diagonal (investigate these)
- Spread: Wider spread around diagonal = higher imputation uncertainty

**Labels on Points:**
Format is "City-Month" to help identify problematic imputations. If certain cities or months consistently have larger errors, this might indicate:
- Those cities are unusual (outliers)
- Those months are harder to predict from others
- More principal components might be needed

In [62]:
# Visualize imputation convergence and quality
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Convergence History', 'Imputed vs Actual Values'
])

# Convergence plot
fig.add_trace(go.Scatter(y=convergence, mode='lines+markers',
                         marker=dict(size=6), line=dict(color='steelblue')),
              row=1, col=1)

# Actual vs Imputed scatter
actuals = [r['Actual'] for r in imputation_results]
imputed = [r['Imputed'] for r in imputation_results]
labels = [f"{r['City']}-{r['Month']}" for r in imputation_results]

fig.add_trace(go.Scatter(
    x=actuals, y=imputed, mode='markers+text',
    text=labels, textposition='top center', textfont=dict(size=8),
    marker=dict(size=10, color='green')
), row=1, col=2)

# Add perfect line
min_val, max_val = min(actuals + imputed), max(actuals + imputed)
fig.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', line=dict(dash='dash', color='red'),
    name='Perfect'
), row=1, col=2)

fig.update_layout(height=400, width=1000, showlegend=False)
fig.update_xaxes(title_text='Iteration', row=1, col=1)
fig.update_xaxes(title_text='Actual Temperature (°C)', row=1, col=2)
fig.update_yaxes(title_text='Change (RMSE)', row=1, col=1)
fig.update_yaxes(title_text='Imputed Temperature (°C)', row=1, col=2)
fig.show()

---
## Part 8: Clustering in PC Space

### Cell 26: Cluster Evaluation Metrics

This cell uses PC scores as features for clustering and evaluates different numbers of clusters using three metrics.

**Why Cluster in PC Space?**
1. **Dimensionality reduction**: Reduces from 12 variables to 2-3 PCs
2. **Noise reduction**: PCA filters out noise in lower components
3. **Uncorrelated features**: PCs are orthogonal, improving clustering algorithm performance
4. **Interpretability**: Clusters can be visualized in 2D PC space

**Three Evaluation Metrics:**

**1. Elbow Method (Inertia)**
- Inertia = Within-cluster sum of squared distances to centroid
- Look for the "elbow" where adding more clusters gives diminishing returns
- Lower is better, but we want parsimony

**2. Silhouette Score**
$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$
- $a_i$ = average distance to points in same cluster
- $b_i$ = average distance to points in nearest other cluster
- Range: [-1, 1], higher is better
- Measures cluster cohesion and separation

**3. Calinski-Harabasz Index**
$$CH = \frac{B/(k-1)}{W/(n-k)}$$
- B = between-cluster variance, W = within-cluster variance
- Higher is better
- Ratio of separation to compactness

**Choosing k:**
- All three metrics should agree (or at least not strongly disagree)
- Domain knowledge matters: Do the clusters make geographic/climatic sense?
- For this data: Expect 3-5 natural clusters (Mediterranean, Continental, Maritime, Nordic, etc.)

In [63]:
# Use PC scores for clustering (dimensionality reduction before clustering)
scores_for_clustering = scores[:, :3]  # First 3 PCs

# Determine optimal number of clusters
k_range = range(2, 8)
silhouette_scores = []
calinski_scores = []
inertias = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scores_for_clustering)
    silhouette_scores.append(silhouette_score(scores_for_clustering, labels))
    calinski_scores.append(calinski_harabasz_score(scores_for_clustering, labels))
    inertias.append(kmeans.inertia_)

# Plot cluster evaluation metrics
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'Elbow Method (Inertia)', 'Silhouette Score', 'Calinski-Harabasz Index'
])

fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers'), row=1, col=1)
fig.add_trace(go.Scatter(x=list(k_range), y=silhouette_scores, mode='lines+markers'), row=1, col=2)
fig.add_trace(go.Scatter(x=list(k_range), y=calinski_scores, mode='lines+markers'), row=1, col=3)

fig.update_layout(height=300, width=1100, showlegend=False,
                  title_text='Cluster Evaluation Metrics')
fig.update_xaxes(title_text='Number of Clusters')
fig.show()

optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"\nOptimal k by silhouette: {optimal_k}")


Optimal k by silhouette: 3


### Cell 27: Applying Clustering Algorithms

This cell applies two different clustering algorithms to the PC scores:

**K-Means Clustering**
- Partitions data into exactly $k$ clusters
- Minimizes within-cluster variance
- Assumes spherical clusters of similar size
- Fast and scalable
- Sensitive to initialization (we use `n_init=10` for stability)

**Hierarchical (Agglomerative) Clustering**
- Builds a tree (dendrogram) of clusters
- "Ward" linkage minimizes variance increase when merging clusters
- Can reveal hierarchical structure in data
- More stable than K-Means (no random initialization)
- Better for non-spherical clusters

**Why Compare Both?**
- Different algorithms may reveal different structures
- Agreement between methods increases confidence in results
- Disagreement prompts investigation of alternative structures

**Storing Results:**
We add cluster labels to our results DataFrame for further analysis:
- `KMeans_Cluster`: Cluster assignment from K-Means
- `Hierarchical_Cluster`: Cluster assignment from hierarchical clustering

**Choice of k=4:**
While optimal k by silhouette might suggest a different number, we use k=4 based on domain knowledge (geographic climate zones). This balances statistical criteria with interpretability.

In [64]:
# Perform clustering with optimal k
k = 4  # Can adjust based on domain knowledge
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(scores_for_clustering)

# Also try hierarchical clustering
hierarchical = AgglomerativeClustering(n_clusters=k, linkage='ward')
hier_labels = hierarchical.fit_predict(scores_for_clustering)

results_df['KMeans_Cluster'] = cluster_labels
results_df['Hierarchical_Cluster'] = hier_labels

### Cell 28: Visualizing Clusters in PC Space

This cell creates side-by-side scatter plots showing cluster assignments from both algorithms in the PC1-PC2 plane.

**Plot Structure:**
- **Left panel**: K-Means clustering results
- **Right panel**: Hierarchical clustering results
- Each point is a city, colored by cluster membership
- Text labels identify each city

**What to Look For:**

**Cluster Separation:**
- Well-separated clusters suggest distinct climate types
- Overlapping clusters may indicate gradual transitions between climate zones

**Consistency Between Methods:**
- If both methods assign the same cities to the same clusters → robust structure
- Different assignments → the boundary between groups is ambiguous

**Cluster Positions in PC Space:**
Using our previous interpretation (PC1=temperature level, PC2=seasonality):
- **Upper-left quadrant**: Cold + continental (Nordic/Eastern European)
- **Upper-right quadrant**: Warm + continental (rare in Europe)
- **Lower-left quadrant**: Cold + maritime (Northern Atlantic coast)
- **Lower-right quadrant**: Warm + maritime (Mediterranean)

**Color Coding:**
Each cluster gets a distinct color from the Set1 palette. The same colors are used in both panels for comparison, but note that cluster numbering may differ between methods (Cluster 1 in K-Means might correspond to Cluster 3 in Hierarchical).

In [65]:
# Visualize clusters in PC space
fig = make_subplots(rows=1, cols=2, subplot_titles=['K-Means Clustering', 'Hierarchical Clustering'])

colors_kmeans = px.colors.qualitative.Set1[:k]

for cluster in range(k):
    mask = cluster_labels == cluster
    cluster_cities = [cities[i] for i in range(len(cities)) if mask[i]]
    
    fig.add_trace(go.Scatter(
        x=scores[mask, 0], y=scores[mask, 1],
        mode='markers+text', text=cluster_cities,
        textposition='top center', textfont=dict(size=8),
        marker=dict(size=12, color=colors_kmeans[cluster]),
        name=f'Cluster {cluster + 1}'
    ), row=1, col=1)
    
    mask_h = hier_labels == cluster
    cluster_cities_h = [cities[i] for i in range(len(cities)) if mask_h[i]]
    
    fig.add_trace(go.Scatter(
        x=scores[mask_h, 0], y=scores[mask_h, 1],
        mode='markers+text', text=cluster_cities_h,
        textposition='top center', textfont=dict(size=8),
        marker=dict(size=12, color=colors_kmeans[cluster]),
        name=f'Cluster {cluster + 1}', showlegend=False
    ), row=1, col=2)

fig.update_layout(height=500, width=1100)
fig.update_xaxes(title_text='PC1', row=1, col=1)
fig.update_xaxes(title_text='PC1', row=1, col=2)
fig.update_yaxes(title_text='PC2', row=1, col=1)
fig.update_yaxes(title_text='PC2', row=1, col=2)
fig.show()

### Cell 29: Cluster Profiles - Temperature Patterns by Cluster

This cell calculates and visualizes the **mean temperature profile** for each cluster, showing the characteristic seasonal pattern of cities in that group.

**What is a Cluster Profile?**
The average temperature for each month across all cities in a cluster:
$$\text{Profile}_k = \frac{1}{n_k}\sum_{i \in \text{Cluster } k} \mathbf{x}_i$$

**Interpretation:**
Each line represents a "typical" city in that cluster:
- **Vertical position**: Overall temperature level (relates to PC1)
- **Amplitude**: Seasonal variation (relates to PC2)
- **Shape**: Timing and symmetry of seasons

**Expected Cluster Types:**

1. **Mediterranean Cluster**: High overall temps, moderate seasonality
2. **Continental Cluster**: Moderate temps, high seasonality (cold winters, warm summers)
3. **Maritime Cluster**: Mild temps, low seasonality (cool summers, mild winters)
4. **Nordic Cluster**: Low overall temps, moderate to high seasonality

**Cluster Membership:**
The text output lists which cities belong to each cluster, along with:
- Average latitude of cluster members
- Average temperature of cluster members

This allows validation: Do the clusters make geographic sense? Are similar cities grouped together?

In [66]:
# Cluster profiles: average temperature pattern per cluster
cluster_profiles = []
for cluster in range(k):
    mask = cluster_labels == cluster
    profile = X_raw[mask].mean(axis=0)
    cluster_profiles.append(profile)

fig = go.Figure()
for cluster, profile in enumerate(cluster_profiles):
    n_cities = (cluster_labels == cluster).sum()
    fig.add_trace(go.Scatter(
        x=month_cols, y=profile, mode='lines+markers',
        name=f'Cluster {cluster + 1} (n={n_cities})',
        line=dict(width=3)
    ))

fig.update_layout(
    title='Average Temperature Profile by Cluster',
    xaxis_title='Month', yaxis_title='Temperature (°C)',
    height=450, width=900
)
fig.show()

# Show cluster membership
print("\nCluster Membership:")
for cluster in range(k):
    mask = cluster_labels == cluster
    cluster_cities = [cities[i] for i in range(len(cities)) if mask[i]]
    avg_lat = latitudes[mask].mean()
    avg_temp = X_raw[mask].mean()
    print(f"\nCluster {cluster + 1} (n={len(cluster_cities)}):")
    print(f"  Cities: {', '.join(cluster_cities)}")
    print(f"  Avg Latitude: {avg_lat:.1f}°N, Avg Temp: {avg_temp:.1f}°C")


Cluster Membership:

Cluster 1 (n=5):
  Cities: Helsinki, Moscow, Oslo, Stockholm, Edinburgh
  Avg Latitude: 59.0°N, Avg Temp: 5.4°C

Cluster 2 (n=11):
  Cities: Ankara, Belgrade, Bucarest, Budapest, London, Paris, Rome, Sofia, Madrid, Milan, Skopje
  Avg Latitude: 44.5°N, Avg Temp: 12.9°C

Cluster 3 (n=14):
  Cities: Amsterdam, Berlin, Bern, Bonn, Bruxells, Copenhagen, Dublin, Luxemborg, Prague, Vienna, Warsaw, Bratislava, St. Petersberg, Ljubljana
  Avg Latitude: 50.9°N, Avg Temp: 9.7°C

Cluster 4 (n=3):
  Cities: Athens, Lisbon, Marbella
  Avg Latitude: 37.7°N, Avg Temp: 19.5°C


---
## Part 9: Orthogonal Decomposition - Independent Sources of Variation

### Cell 30: Verifying Orthogonality of Principal Components

This cell mathematically verifies two fundamental properties of PCA: **orthonormality of loadings** and **uncorrelatedness of scores**.

**Property 1: Orthonormal Loadings**
The loading vectors (eigenvectors) should satisfy:
$$\mathbf{P}^T\mathbf{P} = \mathbf{I}$$

This means:
- Each loading vector has unit length: $\|\mathbf{p}_k\| = 1$ (diagonal = 1)
- Loading vectors are perpendicular: $\mathbf{p}_i^T\mathbf{p}_j = 0$ for $i \neq j$ (off-diagonal = 0)

**Property 2: Uncorrelated Scores**
The score vectors should satisfy:
$$\text{Corr}(\mathbf{t}_i, \mathbf{t}_j) = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases}$$

This means principal components capture **independent** sources of variation.

**Why Orthogonality Matters:**

1. **No redundancy**: Each PC captures unique information
2. **Interpretability**: Effects can be attributed to individual components without confounding
3. **Variance additivity**: Total variance = sum of individual PC variances
4. **Regression stability**: PC scores as predictors avoid multicollinearity

**Numerical Precision:**
The matrices should be very close to identity. Small deviations (e.g., $10^{-15}$) are due to floating-point arithmetic and are negligible.

In [67]:
# Verify orthogonality of principal components
print("Component Orthogonality Check (loadings)")
print("-" * 50)

# Loadings should be orthonormal
loadings_inner_product = loadings.T @ loadings
print("\nLoadings inner product matrix (should be identity):")
print(pd.DataFrame(loadings_inner_product[:4, :4].round(6),
                   index=[f'PC{i+1}' for i in range(4)],
                   columns=[f'PC{i+1}' for i in range(4)]))

# Scores should be uncorrelated
scores_corr = np.corrcoef(scores.T)
print("\nScores correlation matrix (should be identity):")
print(pd.DataFrame(scores_corr[:4, :4].round(6),
                   index=[f'PC{i+1}' for i in range(4)],
                   columns=[f'PC{i+1}' for i in range(4)]))

Component Orthogonality Check (loadings)
--------------------------------------------------

Loadings inner product matrix (should be identity):
     PC1  PC2  PC3  PC4
PC1  1.0 -0.0  0.0 -0.0
PC2 -0.0  1.0  0.0 -0.0
PC3  0.0  0.0  1.0  0.0
PC4 -0.0 -0.0  0.0  1.0

Scores correlation matrix (should be identity):
     PC1  PC2  PC3  PC4
PC1  1.0 -0.0  0.0 -0.0
PC2 -0.0  1.0  0.0 -0.0
PC3  0.0  0.0  1.0  0.0
PC4 -0.0 -0.0  0.0  1.0


### Cell 31: Variance Decomposition by Original Variable

This cell computes how much of each **original variable's** variance is explained by each principal component.

**The Question:**
For each month, what percentage of its variance comes from PC1, PC2, etc.?

**Mathematical Formula:**
The variance of variable $j$ explained by component $k$ is:
$$\text{Var}_{jk} = l_{jk}^2 \times \text{Var}(PC_k)$$

where $l_{jk}$ is the loading of variable $j$ on component $k$.

The percentage is:
$$\%\text{Var}_{jk} = \frac{l_{jk}^2 \times \text{Var}(PC_k)}{\text{Var}(X_j)} \times 100$$

**Reading the Table:**
- Each row is an original variable (month)
- Each column is a principal component (or residual)
- Values are percentages that sum to 100% across each row

**Interpretation:**
- If January has 90% from PC1 and 8% from PC2 → January is almost entirely explained by the first two components
- The "Residual" column shows unexplained variance (ideally small)
- Variables with high residuals are poorly represented by the retained components

**Why This Matters:**
- Identifies which variables are well-captured by the PCA model
- Variables with high residuals might need more components or special treatment
- Useful for understanding the structure of your data at the variable level

In [68]:
# Decompose total variance into orthogonal components
def variance_decomposition(X, loadings, scores, feature_names):
    """
    Decompose variance of each original variable into PC contributions.
    """
    n_features = X.shape[1]
    n_components = min(loadings.shape[1], 6)  # Show first 6
    
    decomposition = np.zeros((n_features, n_components + 1))
    
    for j in range(n_features):
        total_var = np.var(X[:, j])
        for k in range(n_components):
            # Variance explained by PC k for variable j
            var_explained = loadings[j, k] ** 2 * np.var(scores[:, k])
            decomposition[j, k] = var_explained / total_var * 100
        decomposition[j, -1] = 100 - decomposition[j, :-1].sum()  # Residual
    
    cols = [f'PC{i+1}' for i in range(n_components)] + ['Residual']
    return pd.DataFrame(decomposition, index=feature_names, columns=cols)

var_decomp = variance_decomposition(X, loadings, scores, month_cols)
print("Variance Decomposition by Variable (%)")
print("="*60)
print(var_decomp.round(1).to_string())

Variance Decomposition by Variable (%)
      PC1   PC2  PC3  PC4  PC5  PC6  Residual
Jan  78.0  19.4  0.8  0.0  0.5  0.5       0.8
Feb  82.2  15.7  0.2  0.6  0.6  0.1       0.7
Mar  88.9   7.1  0.4  2.1  0.0  0.7       0.8
Apr  92.1   0.1  5.1  1.7  0.3  0.1       0.7
May  81.8  13.0  2.9  0.1  0.7  0.0       1.4
Jun  76.1  13.9  8.2  0.7  0.3  0.3       0.5
Jul  74.1  20.1  4.6  0.2  0.0  0.4       0.7
Aug  80.2  13.7  0.8  1.3  3.4  0.3       0.2
Sep  95.6   1.7  1.0  0.0  0.1  0.1       1.4
Oct  97.9   0.0  0.3  0.4  0.3  0.4       0.8
Nov  93.5   3.7  0.0  0.9  0.5  0.6       0.7
Dec  79.8  13.8  1.6  2.7  0.4  1.5       0.2


### Cell 32: Stacked Bar Chart of Variance Decomposition

This cell visualizes the variance decomposition table as a **stacked bar chart**, making it easier to see patterns across variables.

**Chart Structure:**
- X-axis: Original variables (months)
- Y-axis: Percentage of variance (0-100%)
- Stacked bars: Each color represents a different principal component
- Gray section: Residual variance (unexplained)

**What to Look For:**

**Uniform Pattern (PC1 dominates):**
If PC1 (first color) takes up most of every bar uniformly → strong first component structure

**Variable Patterns:**
- PC1 might explain more of winter months (when variance is higher)
- PC2 might explain more of summer months (when seasonality varies more)

**Residuals:**
- Small gray sections → PCA model captures the data well
- Large gray sections → that variable has unique variance not captured

**Month-Specific Insights:**
- Transition months (Apr, Oct) might have more residual variance
- Peak months (Jan, Jul) might be better explained by the main components

This visualization connects the abstract PCA model back to the original variables in an intuitive way.

In [69]:
# Visualize variance decomposition
fig = go.Figure()

colors = px.colors.qualitative.Set2[:6] + ['lightgray']
x_vars = month_cols

for i, col in enumerate(var_decomp.columns):
    fig.add_trace(go.Bar(
        name=col, x=x_vars, y=var_decomp[col],
        marker_color=colors[i]
    ))

fig.update_layout(
    barmode='stack',
    title='Variance Decomposition: PC Contribution to Each Variable',
    xaxis_title='Month', yaxis_title='Variance Explained (%)',
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

### Cell 33: Orthogonal Reconstruction Demonstration

This cell demonstrates how PCA decomposes each observation into **independent (orthogonal) contributions** from each principal component.

**Data Reconstruction Formula:**
$$\hat{\mathbf{x}}_i = \bar{\mathbf{x}} + \sum_{k=1}^{K} t_{ik} \mathbf{p}_k \cdot \sigma$$

Where:
- $\bar{\mathbf{x}}$ = mean of original data
- $t_{ik}$ = score of observation $i$ on component $k$
- $\mathbf{p}_k$ = loading vector for component $k$
- $\sigma$ = standard deviation (for back-transformation)

**Four Panels Explained:**

1. **Original Data**: The actual temperature profiles for selected cities

2. **Reconstruction (2 PCs)**: Data reconstructed using only PC1 and PC2
   - Should closely match original if 2 PCs capture most variance
   - Differences are the residuals (unexplained variation)

3. **PC1 Contribution Only**: What each city looks like with only the "average level" component
   - Flat or nearly flat lines at different heights
   - Represents the overall warmth/coldness of each city

4. **PC2 Contribution Only**: What each city looks like with only the "seasonality" component
   - Shows summer-winter contrast
   - Amplitude varies by city's seasonal intensity

**Key Insight:**
The orthogonality of PCA means:
$$\text{Original} \approx \text{Mean} + \text{PC1 contribution} + \text{PC2 contribution} + \text{Residual}$$

These contributions are additive and independent—changing one doesn't affect the others.

In [70]:
# Reconstruct data using different numbers of components to show orthogonal contributions
fig = make_subplots(rows=2, cols=2, subplot_titles=[
    'Original Data', 'Reconstruction (2 PCs)',
    'PC1 Contribution Only', 'PC2 Contribution Only'
])

# Pick a few cities to visualize
sample_cities = ['Athens', 'Berlin', 'Dublin', 'Madrid']
sample_indices = [cities.index(c) for c in sample_cities]
colors_city = px.colors.qualitative.Plotly[:4]

# Reconstruct with different components
X_recon_2pc = (scores[:, :2] @ loadings[:, :2].T) * scaler.scale_ + scaler.mean_
X_pc1_only = (scores[:, 0:1] @ loadings[:, 0:1].T) * scaler.scale_ + scaler.mean_
X_pc2_only = (scores[:, 1:2] @ loadings[:, 1:2].T) * scaler.scale_ + scaler.mean_

for i, (idx, city) in enumerate(zip(sample_indices, sample_cities)):
    fig.add_trace(go.Scatter(x=month_cols, y=X_raw[idx], mode='lines+markers',
                             name=city, line=dict(color=colors_city[i])), row=1, col=1)
    fig.add_trace(go.Scatter(x=month_cols, y=X_recon_2pc[idx], mode='lines+markers',
                             name=city, line=dict(color=colors_city[i]), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=month_cols, y=X_pc1_only[idx], mode='lines+markers',
                             name=city, line=dict(color=colors_city[i]), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=month_cols, y=X_pc2_only[idx], mode='lines+markers',
                             name=city, line=dict(color=colors_city[i]), showlegend=False), row=2, col=2)

fig.update_layout(height=600, width=1000, title_text='Orthogonal Decomposition of Temperature Profiles')
fig.update_yaxes(title_text='Temperature (°C)')
fig.show()

print("\nInterpretation:")
print("- PC1 contribution captures the overall temperature level (flat offset)")
print("- PC2 contribution captures the seasonality pattern (summer-winter contrast)")
print("- These are mathematically independent (orthogonal) sources of variation")


Interpretation:
- PC1 contribution captures the overall temperature level (flat offset)
- PC2 contribution captures the seasonality pattern (summer-winter contrast)
- These are mathematically independent (orthogonal) sources of variation


---
## Part 10: Leave-One-Out Cross-Validation

### Cell 34: Leave-One-Out Cross-Validation Function

This cell implements **Leave-One-Out Cross-Validation (LOO-CV)** for PCA to assess model stability and identify influential observations.

**What is LOO-CV?**
For each of $n$ observations:
1. Remove that observation from the dataset
2. Fit PCA on the remaining $n-1$ observations  
3. Use the fitted model to reconstruct the held-out observation
4. Measure reconstruction error and model change

**Outputs Computed:**

1. **`prediction_errors`**: How well can the model (trained without this city) reconstruct this city?
   - High error = City is unusual, doesn't fit the general pattern
   - This is similar to Q-residual but uses out-of-sample prediction

2. **`loading_changes`**: How much do the loadings change when this city is removed?
   - Measured by Frobenius norm of loading difference
   - High change = City is influential (removing it changes the PCA structure)

3. **`loo_eigenvalues`**: Eigenvalues for each LOO model
   - Variance in eigenvalues shows model stability

**Sign Alignment:**
Eigenvectors can flip sign (multiply by -1) and still be valid. We align signs before comparing loadings to avoid artificial differences.

**Why LOO-CV Matters:**
1. **Identifies outliers**: Cities with high prediction errors don't fit the model
2. **Identifies influential points**: Cities that strongly affect the PCA structure
3. **Assesses robustness**: If results change dramatically with single observations removed, the model is unstable

In [71]:
def leave_one_out_pca(X, n_components=2):
    """
    Perform LOO-CV for PCA to assess model stability and observation influence.
    
    Returns:
    - loo_loadings: Loadings matrix for each LOO model
    - loo_eigenvalues: Eigenvalues for each LOO model
    - prediction_errors: Reconstruction error when predicting held-out observation
    - influence: Measure of how much each observation affects the model
    """
    n_samples, n_features = X.shape
    
    # Full model for reference
    scaler_full = StandardScaler()
    X_full = scaler_full.fit_transform(X)
    pca_full = PCA(n_components=n_components)
    pca_full.fit(X_full)
    
    loo_loadings = []
    loo_eigenvalues = []
    prediction_errors = []
    loading_changes = []
    
    for i in range(n_samples):
        # Leave out observation i
        X_train = np.delete(X, i, axis=0)
        X_test = X[i:i+1]
        
        # Fit on training data
        scaler_loo = StandardScaler()
        X_train_scaled = scaler_loo.fit_transform(X_train)
        
        pca_loo = PCA(n_components=n_components)
        pca_loo.fit(X_train_scaled)
        
        loo_loadings.append(pca_loo.components_.T)
        loo_eigenvalues.append(pca_loo.explained_variance_)
        
        # Predict held-out observation
        X_test_scaled = scaler_loo.transform(X_test)
        X_test_recon = pca_loo.inverse_transform(pca_loo.transform(X_test_scaled))
        X_test_recon_original = scaler_loo.inverse_transform(X_test_recon)
        
        pred_error = np.sqrt(np.mean((X_test - X_test_recon_original) ** 2))
        prediction_errors.append(pred_error)
        
        # Loading change (align signs and compute Frobenius norm of difference)
        loadings_loo = pca_loo.components_.T
        loadings_full = pca_full.components_.T
        
        # Align signs
        for k in range(n_components):
            if np.dot(loadings_loo[:, k], loadings_full[:, k]) < 0:
                loadings_loo[:, k] *= -1
        
        change = np.linalg.norm(loadings_loo - loadings_full, 'fro')
        loading_changes.append(change)
    
    return {
        'loadings': np.array(loo_loadings),
        'eigenvalues': np.array(loo_eigenvalues),
        'prediction_errors': np.array(prediction_errors),
        'loading_changes': np.array(loading_changes),
        'full_loadings': pca_full.components_.T,
        'full_eigenvalues': pca_full.explained_variance_
    }

# Run LOO analysis
print("Running Leave-One-Out Cross-Validation...")
loo_results = leave_one_out_pca(X_raw, n_components=2)
print("Done!")

Running Leave-One-Out Cross-Validation...
Done!


### Cell 35: LOO Results Visualization - Prediction Error and Influence

This cell visualizes the Leave-One-Out analysis results with two bar charts.

**Left Panel: LOO Prediction Error (Model Fit)**
- Each bar represents a city
- Height = RMSE when predicting that city using a model trained without it
- Sorted from highest to lowest error
- Red dashed line = threshold (mean + 2 standard deviations)

**Interpretation:**
- Cities above the threshold don't fit the general European temperature pattern
- These cities might have unique microclimates or data quality issues
- High prediction error ≈ High Q-residual (different calculation, similar meaning)

**Right Panel: LOO Loading Change (Influence)**
- Each bar represents a city
- Height = Frobenius norm of loading matrix change when that city is removed
- Sorted from highest to lowest change

**Interpretation:**
- Cities above the threshold significantly influence the PCA structure
- Removing influential cities would change the interpretation of PCs
- Influential cities often lie at extreme positions in PC space

**The Difference:**
- **High prediction error**: "This city is weird"
- **High loading change**: "This city defines what's normal"

A city can be:
- Normal AND not influential (typical city)
- Unusual but not influential (outlier that doesn't affect the overall pattern)
- Normal but influential (extreme on main pattern, anchors the PC direction)
- Unusual AND influential (problematic—may be distorting the analysis)

In [72]:
# Add LOO results to dataframe
results_df['LOO_Pred_Error'] = loo_results['prediction_errors']
results_df['LOO_Loading_Change'] = loo_results['loading_changes']

# Visualize LOO results
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'LOO Prediction Error (Model Fit)', 'LOO Loading Change (Influence)'
])

# Sort by prediction error
sorted_df = results_df.sort_values('LOO_Pred_Error', ascending=False)

fig.add_trace(go.Bar(
    x=sorted_df['City'], y=sorted_df['LOO_Pred_Error'],
    marker_color='steelblue'
), row=1, col=1)

# Add threshold line (mean + 2*std)
threshold = results_df['LOO_Pred_Error'].mean() + 2 * results_df['LOO_Pred_Error'].std()
fig.add_hline(y=threshold, line_dash='dash', line_color='red', row=1, col=1)

# Sort by loading change
sorted_df2 = results_df.sort_values('LOO_Loading_Change', ascending=False)

fig.add_trace(go.Bar(
    x=sorted_df2['City'], y=sorted_df2['LOO_Loading_Change'],
    marker_color='orange'
), row=1, col=2)

threshold2 = results_df['LOO_Loading_Change'].mean() + 2 * results_df['LOO_Loading_Change'].std()
fig.add_hline(y=threshold2, line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(height=450, width=1200, showlegend=False)
fig.update_xaxes(tickangle=45)
fig.update_yaxes(title_text='RMSE (°C)', row=1, col=1)
fig.update_yaxes(title_text='Frobenius Norm', row=1, col=2)
fig.show()

### Cell 36: Eigenvalue Stability Analysis

This cell examines how stable the eigenvalues are across all Leave-One-Out models.

**Box Plot Interpretation:**
- Each box represents the distribution of eigenvalue estimates for one principal component across all 33 LOO models
- The box shows the interquartile range (IQR: 25th to 75th percentile)
- The line inside the box is the median
- Whiskers extend to the most extreme points within 1.5×IQR
- Individual points show all 33 LOO estimates (jittered for visibility)
- Red star = eigenvalue from the full model (all data)

**What to Look For:**

**Narrow Boxes = Stable Eigenvalues:**
- The component's variance is robust to removing individual observations
- We can trust this eigenvalue estimate

**Wide Boxes = Unstable Eigenvalues:**
- The component's importance varies depending on which observations are included
- May indicate the component is driven by a few influential observations

**Red Star Position:**
- Should be near the center of the box
- If far from center, the full model may be influenced by specific outliers

**PC-Specific Patterns:**
- PC1 typically most stable (largest eigenvalue, most data supporting it)
- Higher PCs may show more variability (smaller eigenvalues, more sensitive to noise)

**Stability Threshold:**
If the coefficient of variation (CV = std/mean) of eigenvalues is < 10%, the component is generally considered stable.

In [73]:
# Stability of eigenvalues across LOO models
fig = go.Figure()

# Box plots of LOO eigenvalues
for k in range(min(4, loo_results['eigenvalues'].shape[1])):
    fig.add_trace(go.Box(
        y=loo_results['eigenvalues'][:, k],
        name=f'PC{k+1}',
        boxpoints='all',
        jitter=0.3,
        pointpos=0
    ))
    
    # Add full model eigenvalue
    fig.add_trace(go.Scatter(
        x=[f'PC{k+1}'], y=[loo_results['full_eigenvalues'][k]],
        mode='markers', marker=dict(size=15, color='red', symbol='star'),
        name='Full Model' if k == 0 else None, showlegend=(k == 0)
    ))

fig.update_layout(
    title='Eigenvalue Stability: LOO Distributions vs Full Model',
    xaxis_title='Principal Component',
    yaxis_title='Eigenvalue',
    height=450, width=700
)
fig.show()

### Cell 37: LOO Analysis Summary Report

This cell generates a summary report identifying cities that are **outliers** (high prediction error) or **influential** (high loading change) based on the LOO analysis.

**Classification Criteria:**
Using mean + 2 standard deviations as threshold:
- **High prediction error cities**: Outliers with unusual patterns
- **High loading change cities**: Influential observations that shape the PCA model

**Interpretation Guidelines:**

**Cities with High Prediction Error:**
- Their temperature patterns deviate from the European norm
- May have unique geographic features (mountains, coastal effects, urban heat islands)
- Could indicate data quality issues (wrong location, measurement errors)
- Worth investigating individually

**Influential Cities:**
- These observations "anchor" the principal component directions
- Often located at extremes of the PC space
- Removing them would change the interpretation of PC1 or PC2
- Important for model stability assessment

**Combined Assessment:**
- Cities that are both unusual AND influential are most concerning
- They might be distorting the PCA structure
- Consider sensitivity analysis: What happens if we exclude them?

**Practical Implications:**
- If your conclusions depend heavily on a few influential cities, be cautious
- Report influential observations when publishing results
- Consider robust PCA methods if influence is a concern

In [74]:
# Identify influential observations
print("="*60)
print("LEAVE-ONE-OUT ANALYSIS SUMMARY")
print("="*60)

# High prediction error = doesn't fit the model well
high_error_mask = results_df['LOO_Pred_Error'] > threshold
print(f"\nCities with high prediction error (outliers):")
print(f"  {list(results_df[high_error_mask]['City'].values)}")

# High loading change = influential on model structure
high_influence_mask = results_df['LOO_Loading_Change'] > threshold2
print(f"\nInfluential cities (affect model structure):")
print(f"  {list(results_df[high_influence_mask]['City'].values)}")

print("\nInterpretation:")
print("-" * 60)
print("• High prediction error: City's pattern is unusual")
print("• High loading change: Removing this city changes the PCs")
print("• Both high: City is both unusual AND influential")

LEAVE-ONE-OUT ANALYSIS SUMMARY

Cities with high prediction error (outliers):
  ['Vienna']

Influential cities (affect model structure):
  ['Moscow', 'Vienna']

Interpretation:
------------------------------------------------------------
• High prediction error: City's pattern is unusual
• High loading change: Removing this city changes the PCs
• Both high: City is both unusual AND influential


---
## Part 11: Comprehensive Summary & Recommendations

### Cell 38: Comprehensive Summary DataFrame

This cell creates a **master summary table** combining all analysis results into a single DataFrame with interpretive classifications.

**Columns in the Summary:**

| Column | Source | Description |
|--------|--------|-------------|
| City | Original | City name |
| Latitude | Original | Geographic latitude |
| Avg_Temp | Calculated | Mean annual temperature |
| Temp_Range | Calculated | Annual temperature range (seasonality) |
| PC1, PC2 | PCA | Principal component scores |
| T2, Q | Diagnostics | Anomaly detection statistics |
| KMeans_Cluster | Clustering | Cluster assignment |
| LOO_Pred_Error | LOO-CV | Leave-one-out prediction error |
| LOO_Loading_Change | LOO-CV | Influence on model |

**Derived Classifications:**

**Climate_Zone** (based on PC1 quartiles):
- Cold: PC1 < -2 (Northern Europe)
- Cool: -2 ≤ PC1 < 0
- Mild: 0 ≤ PC1 < 2
- Warm: PC1 ≥ 2 (Mediterranean)

**Continentality** (based on PC2):
- Maritime: PC2 < -1 (Atlantic coast influence)
- Transitional: -1 ≤ PC2 ≤ 1
- Continental: PC2 > 1 (inland, far from oceans)

**Anomaly_Status**:
- Normal: Both T² and Q within limits
- T² Outlier: Only T² exceeds limit
- Q Outlier: Only Q exceeds limit
- Both: Both statistics exceed limits

This table provides a complete profile of each city from multiple analytical perspectives.

In [75]:
# Create comprehensive summary table
summary_df = results_df[['City', 'Latitude', 'Avg_Temp', 'Temp_Range', 
                          'PC1', 'PC2', 'T2', 'Q', 'KMeans_Cluster',
                          'LOO_Pred_Error', 'LOO_Loading_Change']].copy()

# Add interpretive columns
summary_df['Climate_Zone'] = pd.cut(summary_df['PC1'], 
                                     bins=[-np.inf, -2, 0, 2, np.inf],
                                     labels=['Cold', 'Cool', 'Mild', 'Warm'])

summary_df['Continentality'] = pd.cut(summary_df['PC2'],
                                        bins=[-np.inf, -1, 1, np.inf],
                                        labels=['Maritime', 'Transitional', 'Continental'])

summary_df['Anomaly_Status'] = 'Normal'
summary_df.loc[summary_df['T2'] > diag['T2_limit'], 'Anomaly_Status'] = 'T² Outlier'
summary_df.loc[summary_df['Q'] > diag['Q_limit'], 'Anomaly_Status'] = 'Q Outlier'
summary_df.loc[(summary_df['T2'] > diag['T2_limit']) & 
               (summary_df['Q'] > diag['Q_limit']), 'Anomaly_Status'] = 'Both'

print("Comprehensive City Analysis:")
print("="*100)
display_cols = ['City', 'Latitude', 'Climate_Zone', 'Continentality', 
                'KMeans_Cluster', 'Anomaly_Status']
print(summary_df[display_cols].sort_values('Latitude', ascending=False).to_string(index=False))

Comprehensive City Analysis:
          City  Latitude Climate_Zone Continentality  KMeans_Cluster Anomaly_Status
      Helsinki     60.17         Cold   Transitional               0         Normal
          Oslo     59.95         Cold   Transitional               0         Normal
     Edinburgh     59.56         Cold   Transitional               0         Normal
     Stockholm     59.32         Cold   Transitional               0         Normal
St. Petersberg     55.95         Cool       Maritime               2         Normal
        Moscow     55.75         Cold    Continental               0     T² Outlier
    Copenhagen     55.67         Cold   Transitional               2         Normal
        Dublin     53.33         Cold       Maritime               2         Normal
        Berlin     52.52         Cool   Transitional               2         Normal
     Amsterdam     52.37         Cool       Maritime               2         Normal
        Warsaw     52.23         Cool   Transit

### Cell 39: Final Interactive Summary Visualization

This cell creates a comprehensive **interactive scatter plot** that encodes multiple dimensions of information simultaneously.

**Visual Encodings:**

| Visual Element | Data Dimension |
|----------------|----------------|
| X position | PC1 (overall temperature level) |
| Y position | PC2 (seasonality/continentality) |
| Color | K-Means cluster assignment |
| Size | Temperature range (larger = more seasonal) |
| Text labels | City names |
| Hover data | Latitude, Avg_Temp, Climate_Zone, Continentality, Anomaly_Status |

**Quadrant Annotations:**
The plot is divided into interpretive quadrants:
- **Warm Continental** (upper-right): Hot summers, cold winters, warm average
- **Cool Continental** (upper-left): Hot summers, cold winters, cool average
- **Warm Maritime** (lower-right): Mild year-round, warm average
- **Cool Maritime** (lower-left): Mild year-round, cool average

**How to Use This Plot:**

1. **Explore clusters**: Do colors (clusters) group geographically similar cities?
2. **Check classifications**: Hover over cities to see their full profile
3. **Identify outliers**: Cities in unexpected positions warrant investigation
4. **Validate understanding**: Do the quadrant labels match your geographic knowledge?

**Interactive Features:**
- Hover for detailed city information
- Click legend to toggle cluster visibility
- Zoom and pan to explore specific regions
- Double-click to reset view

In [76]:
# Final interactive summary plot
fig = px.scatter(
    summary_df, x='PC1', y='PC2', color='KMeans_Cluster',
    size='Temp_Range', size_max=20,
    hover_name='City',
    hover_data=['Latitude', 'Avg_Temp', 'Climate_Zone', 'Continentality', 'Anomaly_Status'],
    text='City',
    color_continuous_scale='Viridis',
    title='PCA Summary: European Cities Climate Clustering'
)

fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(
    xaxis_title=f'PC1: Overall Temperature ({explained_var[0]*100:.1f}%)',
    yaxis_title=f'PC2: Seasonality ({explained_var[1]*100:.1f}%)',
    height=700, width=1000
)

# Add quadrant labels
fig.add_annotation(x=3, y=3, text='Warm Continental', showarrow=False, font=dict(size=12))
fig.add_annotation(x=-3, y=3, text='Cool Continental', showarrow=False, font=dict(size=12))
fig.add_annotation(x=3, y=-2, text='Warm Maritime', showarrow=False, font=dict(size=12))
fig.add_annotation(x=-3, y=-2, text='Cool Maritime', showarrow=False, font=dict(size=12))

fig.show()

---
## Summary: The Full Power of PCA

This notebook demonstrated the comprehensive capabilities of PCA:

### 1. **Dimensionality Reduction**
- Reduced 12 monthly temperature features to 2-3 meaningful components
- First 2 PCs capture >95% of variance

### 2. **Insight Generation**
- PC1 = Overall temperature level (latitude effect)
- PC2 = Seasonality/Continentality (distance from ocean)

### 3. **Anomaly Detection**
- Hotelling's T² identifies extreme observations within the model
- Q-residuals identify observations that don't fit the model

### 4. **Data Imputation**
- EM-PCA algorithm successfully imputes missing values
- RMSE < 2°C demonstrates strong reconstruction capability

### 5. **Clustering**
- PCA scores provide excellent features for clustering
- Revealed natural climate groupings (Mediterranean, Continental, etc.)

### 6. **Orthogonal Decomposition**
- PCs represent independent, uncorrelated sources of variation
- Enables clean separation of effects

### 7. **Model Validation (LOO)**
- Identified influential observations
- Confirmed model stability across different subsets

### Key Takeaways
- PCA is not just dimensionality reduction—it's a complete analytical framework
- The combination of scores, loadings, and residuals provides rich diagnostics
- Cross-validation ensures robust, generalizable models

### Cell 40: Exporting Results to Excel

This cell saves all analysis results to an Excel file with multiple sheets for easy sharing and further analysis.

**Excel Sheets Created:**

1. **Summary**: The comprehensive city summary table
   - All cities with their classifications, scores, and diagnostics
   - Good for sorting, filtering, and quick reference

2. **Loadings**: Principal component loadings
   - Shows how each month contributes to each PC
   - Essential for interpreting component meaning

3. **Variance_Explained**: Variance decomposition table
   - Eigenvalues, percentages, cumulative variance
   - Component selection criteria (Kaiser, Parallel)

4. **Variance_Decomposition**: Variable-level variance breakdown
   - How much of each month's variance is explained by each PC
   - Useful for assessing model fit per variable

**Why Export to Excel?**
- Share results with collaborators who don't use Python
- Perform additional analyses in familiar spreadsheet environment
- Create formatted tables for reports and publications
- Archive results for reproducibility

**File Location:**
The file `PCA_Complete_Analysis.xlsx` is saved in the same directory as the notebook.

**Note**: This requires the `openpyxl` package (already installed for reading the input data).

In [77]:
# Save results to Excel for further analysis
output_filename = 'PCA_Complete_Analysis.xlsx'

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    loadings_df.to_excel(writer, sheet_name='Loadings')
    variance_df.to_excel(writer, sheet_name='Variance_Explained', index=False)
    var_decomp.to_excel(writer, sheet_name='Variance_Decomposition')

print(f"✓ Results saved to {output_filename}")

✓ Results saved to PCA_Complete_Analysis.xlsx
